# Notebook version: v5.0

# Working with GitHub

## Git imports

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess

## Git constants

In [ ]:
DRIVE_MOUNT_POINT = Path('/content/drive')
COLAB_REPO_PATH = DRIVE_MOUNT_POINT / 'MyDrive' / 'patternsR-D'
LOCAL_REPO_PATH = Path.cwd()
GIT_REMOTE = 'origin'
GIT_BRANCH = 'main'
NOTEBOOK_FILE = 'doubleTop_DoubleBottom_detection.ipynb'

## Git sync functions

In [ ]:
def run_git(repo_path, *args, capture_output=False, check=True):
    result = subprocess.run(
        ['git', *args],
        cwd=repo_path,
        check=False,
        text=True,
        capture_output=True,
    )
    if not capture_output:
        if result.stdout:
            print(result.stdout, end='')
        if result.stderr:
            print(result.stderr, end='')
    if check and result.returncode != 0:
        command = ' '.join(['git', *args])
        details = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(
            f'Git command failed ({result.returncode}): {command}\n{details}'
        )
    return result


def require_main_branch(repo_path):
    if not (repo_path / '.git').exists():
        raise RuntimeError(f'Git clone не найден: {repo_path}')

    branch = run_git(
        repo_path,
        'branch', '--show-current', capture_output=True
    ).stdout.strip()
    if branch != GIT_BRANCH:
        raise RuntimeError(
            f'Ожидалась ветка {GIT_BRANCH}, но активна {branch!r}.'
        )


def print_sync_status(repo_path, message):
    revision = run_git(
        repo_path,
        'log', '-1', '--format=%h %ad %s', '--date=iso-strict',
        capture_output=True,
    ).stdout.strip()
    print(f'{message}: {revision}')
    run_git(repo_path, 'status', '--short', '--branch')


def ensure_git_identity(repo_path):
    """Проверяет identity commit и при необходимости берёт её из Colab Secrets."""
    name = run_git(
        repo_path, 'config', '--get', 'user.name',
        capture_output=True, check=False,
    ).stdout.strip()
    email = run_git(
        repo_path, 'config', '--get', 'user.email',
        capture_output=True, check=False,
    ).stdout.strip()

    if not name or not email:
        try:
            from google.colab import userdata
            name = name or userdata.get('GIT_USER_NAME')
            email = email or userdata.get('GIT_USER_EMAIL')
        except Exception:
            pass

    if not name or not email:
        raise RuntimeError(
            'Git identity не настроена. Выполни в clone:\n'
            "git config user.name 'Your Name'\n"
            "git config user.email 'you@example.com'\n"
            'или создай Colab Secrets GIT_USER_NAME и GIT_USER_EMAIL.'
        )

    run_git(repo_path, 'config', 'user.name', name)
    run_git(repo_path, 'config', 'user.email', email)


def hard_sync_colab():
    """Монтирует Drive и жёстко синхронизирует Colab clone с origin/main."""
    from google.colab import drive

    if not (DRIVE_MOUNT_POINT / 'MyDrive').is_dir():
        drive.mount(str(DRIVE_MOUNT_POINT))
    if not COLAB_REPO_PATH.is_dir():
        raise FileNotFoundError(f'Git clone не найден: {COLAB_REPO_PATH}')

    require_main_branch(COLAB_REPO_PATH)
    run_git(COLAB_REPO_PATH, 'fetch', GIT_REMOTE, GIT_BRANCH)
    run_git(
        COLAB_REPO_PATH,
        'reset', '--hard', f'{GIT_REMOTE}/{GIT_BRANCH}',
    )
    print_sync_status(COLAB_REPO_PATH, 'Colab синхронизирован')


def push_colab_outputs():
    """Коммитит и отправляет только notebook с Colab outputs."""
    if not COLAB_REPO_PATH.is_dir():
        raise FileNotFoundError(f'Git clone не найден: {COLAB_REPO_PATH}')

    require_main_branch(COLAB_REPO_PATH)
    notebook_path = COLAB_REPO_PATH / NOTEBOOK_FILE
    if not notebook_path.is_file():
        raise FileNotFoundError(f'Notebook не найден: {notebook_path}')

    # Перед вызовом этой функции сначала сохрани notebook.
    run_git(COLAB_REPO_PATH, 'add', '--', NOTEBOOK_FILE)
    staged_diff = run_git(
        COLAB_REPO_PATH,
        'diff', '--cached', '--quiet', '--', NOTEBOOK_FILE,
        check=False,
    )
    if staged_diff.returncode == 0:
        print('Новых изменений notebook для commit нет.')
        return
    if staged_diff.returncode != 1:
        raise RuntimeError('Не удалось проверить staged diff notebook.')

    ensure_git_identity(COLAB_REPO_PATH)
    commit_message = (
        'Save Colab outputs '
        f"{datetime.now(timezone.utc).astimezone().isoformat(timespec='seconds')}"
    )
    run_git(COLAB_REPO_PATH, 'commit', '-m', commit_message)
    run_git(COLAB_REPO_PATH, 'push', GIT_REMOTE, GIT_BRANCH)
    print_sync_status(COLAB_REPO_PATH, 'Outputs отправлены')


def hard_sync_local():
    """
    Жёстко синхронизирует локальный clone с origin/main.

    ВНИМАНИЕ: reset --hard удаляет незакоммиченные изменения tracked-файлов.
    Рабочий порядок полностью совпадает с ручным hard pull:
    fetch --prune origin main -> reset --hard origin/main.
    """
    require_main_branch(LOCAL_REPO_PATH)
    run_git(
        LOCAL_REPO_PATH,
        'fetch', '--prune', GIT_REMOTE, GIT_BRANCH,
    )
    run_git(
        LOCAL_REPO_PATH,
        'reset', '--hard', f'{GIT_REMOTE}/{GIT_BRANCH}',
    )
    print_sync_status(LOCAL_REPO_PATH, 'Локальная копия синхронизирована')


## Hard sync from GitHub

In [ ]:
hard_sync_colab()

## Push Colab outputs to GitHub

In [ ]:
push_colab_outputs()

## Hard sync local working copy

In [ ]:
hard_sync_local()

# Import libraries & set constants

## Downloading needed libraries

In [1]:
!pip install ccxt pandas mplfinance
!pip install boto3
!pip install numba

## Import Libraries

In [2]:
import ccxt
import pandas as pd
from datetime import datetime
import time
import io
import math
from google.colab import drive, userdata
import os
import sys
import gc
import shutil
import tempfile
from pathlib import Path
from dataclasses import dataclass, field
import boto3

import numpy as np
from scipy.signal import find_peaks
from tqdm import tqdm
from tqdm.auto import tqdm
from tqdm.notebook import tqdm
import random
import mplfinance as mpf
from scipy.stats import linregress
import matplotlib.pyplot as plt
import joblib


import xgboost as xgb
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
# ⌐ [AI REVERT]: Removed RandomForestClassifier import (reverting two-stage cascade)
import time
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, precision_recall_curve
from sklearn.metrics import auc, average_precision_score, make_scorer
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit
import joblib
import json
import subprocess
import plotly.graph_objects as go
from numba import njit
import concurrent.futures
import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from sklearn.metrics import precision_recall_curve, average_precision_score

matplotlib.use('Agg')

drive.mount('/content/drive')

# Constant for reproducibility
RANDOM_STATE = 42

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Set constants

In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    symbols: tuple = (
        'BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'XRP/USDT',
        'ADA/USDT', 'DOT/USDT', 'LINK/USDT', 'AVAX/USDT', 'DOGE/USDT',
        'NEAR/USDT', 'ATOM/USDT', 'LTC/USDT', 'PEPE/USDT',
        'SHIB/USDT', 'FET/USDT', 'SUI/USDT', 'APT/USDT', 'OP/USDT',
        'ARB/USDT', 'RENDER/USDT', 'INJ/USDT', 'TIA/USDT'
    )
    class_indices: dict = field(
        default_factory=lambda: {
            'NOISE': 0,
            'DT': 1,
            'DB': 2,
        }
    )
    window_size: int = 50
    train_fraction: float = 0.80
    calibration_fraction_of_holdout: float = 0.50
    tolerance: int = 10
    nms_window: int = 10
    dead_zone: int = 80
    noise_percent: float = 0.90
    target_recall: float = 0.75
    normal_step: int = 2


CONFIG = PipelineConfig()
symbols = list(CONFIG.symbols)
CLASS_INDICES = CONFIG.class_indices
WINDOW_SIZE = CONFIG.window_size
pattern_length = WINDOW_SIZE
cooldown_jump = int(WINDOW_SIZE * 0.33)
normal_step = CONFIG.normal_step


In [4]:
def get_optimal_device():
    """
    Проверяет наличие видеокарты NVIDIA в системе.
    Возвращает 'cuda' если доступно, иначе 'cpu'.
    """
    try:
        # Пытаемся вызвать системную утилиту драйвера видеокарты
        subprocess.check_output('nvidia-smi')
        print("🟢 Ура! Обнаружен GPU (CUDA). Включаем турборежим.")
        return 'cuda'
    except Exception:
        print("🟡 GPU не найден или недоступен. Откатываемся на CPU (процессор).")
        return 'cpu'

# Определяем устройство для всей дальнейшей работы
ACTIVE_DEVICE = get_optimal_device()

🟢 Ура! Обнаружен GPU (CUDA). Включаем турборежим.


In [5]:
# Обратная совместимость для существующих функций pipeline.
TARGET_MAP = CLASS_INDICES

In [6]:
YANDEX_S3_BUCKET = 'tickframe-candidates'
ACCESS_KEY_ID = userdata.get("ACCESS_KEY_ID")
SECRET_ACCESS_KEY = userdata.get("SECRET_ACCESS_KEY")
sample_patterns_prefix = "evaluation_sample"
main_dataset_prefix = "labeled_train_v1"

In [7]:
RANDOM_STATE = int(time.time())

In [8]:
# Единый контракт признаков: исходные 15 признаков для DT/DB.
FEATURE_COLUMNS_NAME = [
    'NATR_14', 'Trend_50', 'Range_Position',
    'H_Idx_1', 'L_Idx_1', 'H_Prc_1', 'L_Prc_1',
    'H_Idx_2', 'L_Idx_2', 'H_Prc_2', 'L_Prc_2',
    'DT_Width', 'DB_Width', 'DT_Symmetry_Prc', 'DB_Symmetry_Prc',
    'DT_Peak_Dominance', 'DB_Valley_Dominance', 'Window_Range_ATR_Pct'
]

In [9]:
def renew_random_state():
    global RANDOM_STATE
    RANDOM_STATE = int(time.time())
    return RANDOM_STATE

# Auto-data-labeling

## Pooling raw 5-minutes candles from google drive

In [ ]:
def isSymbolInFileName(symbols, filename):
    for symbol in symbols:
        if symbol.partition('/')[0] in filename:
            return True
    return False

In [ ]:
raw_input_folder = '/content/drive/MyDrive/Crypto_Raw_Data'
raw_candles = {}

print("Downloading raw data from disk")
for symbol in symbols:
    clean_name = symbol.replace('/', '_') + '_raw.csv'
    path = f'{raw_input_folder}/{clean_name}'
    if os.path.exists(path):
        raw_candles[symbol] = pd.read_csv(path, index_col=0, parse_dates=True)
        print(f"Data: {symbol} ({len(raw_candles[symbol])} строк)")

In [ ]:
display(raw_candles[symbols[0]].head())

## Add smart features to candles

In [10]:
@njit
def _find_extrema_numba(high_windows, low_windows, window_size, min_dist):
    """
    JIT-скомпилированная функция для молниеносного поиска 2-х пиков и 2-х впадин.
    """
    n_windows = len(high_windows)

    macro_high_indices = np.zeros((n_windows, 2))
    macro_high_prices = np.zeros((n_windows, 2))
    macro_low_indices = np.zeros((n_windows, 2))
    macro_low_prices = np.zeros((n_windows, 2))

    for i in range(n_windows):
        h_win = high_windows[i]
        l_win = low_windows[i]

        available_h = np.ones(window_size, dtype=np.bool_)
        available_l = np.ones(window_size, dtype=np.bool_)

        # Ищем 2 пика
        for step in range(2):
            best_h_val = -np.inf
            best_h_idx = -1

            # Вручную находим максимум с учетом доступности
            for j in range(window_size):
                if available_h[j] and h_win[j] > best_h_val:
                    best_h_val = h_win[j]
                    best_h_idx = j

            if best_h_idx == -1:
                break

            macro_high_indices[i, step] = window_size - best_h_idx
            macro_high_prices[i, step] = best_h_val

            # Закрашиваем область вокруг найденного пика (выключаем)
            start_idx_h = max(0, best_h_idx - min_dist)
            end_idx_h = min(window_size, best_h_idx + min_dist + 1)
            for j in range(start_idx_h, end_idx_h):
                available_h[j] = False

        # Ищем 2 впадины
        for step in range(2):
            best_l_val = np.inf
            best_l_idx = -1

            # Вручную находим минимум с учетом доступности
            for j in range(window_size):
                if available_l[j] and l_win[j] < best_l_val:
                    best_l_val = l_win[j]
                    best_l_idx = j

            if best_l_idx == -1:
                break

            macro_low_indices[i, step] = window_size - best_l_idx
            macro_low_prices[i, step] = best_l_val

            # Закрашиваем область вокруг найденной впадины
            start_idx_l = max(0, best_l_idx - min_dist)
            end_idx_l = min(window_size, best_l_idx + min_dist + 1)
            for j in range(start_idx_l, end_idx_l):
                available_l[j] = False

    return macro_high_indices, macro_high_prices, macro_low_indices, macro_low_prices

In [11]:
def add_smart_features(df, window_size=WINDOW_SIZE):
    """
    Единая реализация smart features для DT/DB.

    Функция расположена до первого вызова в auto-labeling pipeline,
    поэтому notebook можно выполнять последовательно сверху вниз.
    """
    # Создаем общий прогресс-бар для этой функции
    with tqdm(total=5, desc="  ↳ Извлечение Smart Features", leave=False) as pbar:
        data = df.copy()

        # =========================================================================
        # 1. БАЗОВЫЕ МЕТРИКИ (ATR и Контекст рынка)
        # =========================================================================
        w_natr = 14
        min_dist = max(2, window_size // 10)

        prev_close = data['Close'].shift(1)
        true_range = pd.concat([
            data['High'] - data['Low'],
            abs(data['High'] - prev_close),
            abs(data['Low'] - prev_close),
        ], axis=1).max(axis=1)
        data[f'NATR_{w_natr}'] = true_range.rolling(w_natr).mean() / data['Close']

        data[f'Trend_{window_size}'] = data['Close'] / data['Close'].shift(window_size) - 1
        min_window = data['Low'].rolling(window_size).min()
        max_window = data['High'].rolling(window_size).max()
        data['Range_Position'] = (data['Close'] - min_window) / (max_window - min_window + 1e-8)

        pbar.update(1) # Шаг 1 выполнен

        # =========================================================================
        # 2. ВЕКТОРНЫЙ ПОИСК ЭКСТРЕМУМОВ (ЧЕРЕЗ NUMBA)
        # =========================================================================
        high_prices = data['High'].values
        low_prices = data['Low'].values

        high_windows = sliding_window_view(high_prices, window_shape=window_size)
        low_windows = sliding_window_view(low_prices, window_shape=window_size)

        # Вызываем скомпилированную функцию (первый вызов займет ~1 сек на компиляцию, остальные пролетят)
        macro_high_indices, macro_high_prices, macro_low_indices, macro_low_prices = \
            _find_extrema_numba(high_windows, low_windows, window_size, min_dist)

        pbar.update(1) # Шаг 2 выполнен

        # =========================================================================
        # 3. ХРОНОЛОГИЧЕСКАЯ СОРТИРОВКА (1=Левый пик, 2=Правый пик)
        # =========================================================================
        sort_idx_h = np.argsort(-macro_high_indices, axis=1)
        macro_high_indices = np.take_along_axis(macro_high_indices, sort_idx_h, axis=1)
        macro_high_prices = np.take_along_axis(macro_high_prices, sort_idx_h, axis=1)

        sort_idx_l = np.argsort(-macro_low_indices, axis=1)
        macro_low_indices = np.take_along_axis(macro_low_indices, sort_idx_l, axis=1)
        macro_low_prices = np.take_along_axis(macro_low_prices, sort_idx_l, axis=1)

        pbar.update(1) # Шаг 3 выполнен

        # =========================================================================
        # 4. ВЫЧИСЛЕНИЕ НОРМАЛИЗОВАННЫХ КООРДИНАТ С УЧЕТОМ ATR
        # =========================================================================
        data = data.iloc[window_size - 1:].copy()
        current_closes = data['Close'].values
        current_atr_usd = data[f'NATR_{w_natr}'].values * current_closes + 1e-8

        for step in range(2):
            data[f'H_Idx_{step+1}'] = macro_high_indices[:, step].astype(int)
            data[f'L_Idx_{step+1}'] = macro_low_indices[:, step].astype(int)

            data[f'H_Prc_{step+1}'] = (macro_high_prices[:, step] - current_closes) / current_atr_usd
            data[f'L_Prc_{step+1}'] = (current_closes - macro_low_prices[:, step]) / current_atr_usd

        pbar.update(1) # Шаг 4 выполнен

        # =========================================================================
        # 5. ОЦИФРОВКА ЭСТЕТИКИ DOUBLE TOP / DOUBLE BOTTOM
        # =========================================================================
        data['DT_Width'] = (data['H_Idx_1'] - data['H_Idx_2']) / window_size
        data['DB_Width'] = (data['L_Idx_1'] - data['L_Idx_2']) / window_size

        data['DT_Symmetry_Prc'] = abs(data['H_Prc_1'] - data['H_Prc_2'])
        data['DB_Symmetry_Prc'] = abs(data['L_Prc_1'] - data['L_Prc_2'])

        data['DT_Peak_Dominance'] = (
            ((macro_high_prices[:, 0] + macro_high_prices[:, 1]) / 2 - current_closes)
            / current_atr_usd
        )
        data['DB_Valley_Dominance'] = (
            (current_closes - (macro_low_prices[:, 0] + macro_low_prices[:, 1]) / 2)
            / current_atr_usd
        )

        rolling_max = data['High'].rolling(window_size).max().values
        rolling_min = data['Low'].rolling(window_size).min().values
        data['Window_Range_ATR_Pct'] = (
            (rolling_max - rolling_min) / current_atr_usd
        ) * 100

        pbar.update(1) # Шаг 5 выполнен

        return data.dropna()

In [ ]:
featured_data = {}
for symbol in tqdm(symbols):
    featured_data[symbol] = add_smart_features(raw_candles[symbol], window_size=WINDOW_SIZE)
display(featured_data[symbols[0]])

## Labeling potential candidates

In [ ]:
def label_dt_db_candidates(df, window_size=WINDOW_SIZE, atr_threshold=0.3, min_width=10, max_width=30, dip_atr_min=2.5, min_dip_pct=0.012, noise_space=0.5):
    """
    РАЗМЕТКА 'GOLDILOCKS' (Target: ~3,500 candidates).
    Идеальный баланс: провал 1.2% + 2.5 ATR, и строгий контроль начала тренда.
    """
    data = df.copy()
    data['Target'] = TARGET_MAP["NOISE"]

    NOISE, DT, DB = TARGET_MAP["NOISE"], TARGET_MAP["DT"], TARGET_MAP["DB"]

    lows = data['Low'].values
    highs = data['High'].values
    closes = data['Close'].values
    natr = data['NATR_14'].values

    h_idx_1 = data['H_Idx_1'].values.astype(int)
    h_idx_2 = data['H_Idx_2'].values.astype(int)
    dt_width = data['DT_Width'].values
    dt_sym = data['DT_Symmetry_Prc'].values

    l_idx_1 = data['L_Idx_1'].values.astype(int)
    l_idx_2 = data['L_Idx_2'].values.astype(int)
    db_width = data['DB_Width'].values
    db_sym = data['DB_Symmetry_Prc'].values

    for i in tqdm(range(len(data)), desc="  ↳ Разметка (Goldilocks)", leave=False):
        if i < window_size + 20:
            continue

        current_atr_usd = natr[i] * closes[i]

        # ==========================================
        # DOUBLE TOP (DT)
        # ==========================================
        if (min_width <= dt_width[i] <= max_width) and (dt_sym[i] <= atr_threshold):

            if h_idx_1[i] >= (window_size - 3):
                continue

            idx_left_peak = i - h_idx_1[i]
            idx_right_peak = i - h_idx_2[i]

            if idx_left_peak >= 0 and idx_left_peak < idx_right_peak:
                valley_low = np.min(lows[idx_left_peak : idx_right_peak])
                avg_peak_high = (highs[idx_left_peak] + highs[idx_right_peak]) / 2
                dip_depth_usd = avg_peak_high - valley_low

                # 1. ЗАЩИТА ОТ ШУМА: Строгий гибрид (2.5 ATR и 1.2% от цены)
                if dip_depth_usd > (dip_atr_min * current_atr_usd) and (dip_depth_usd / closes[i]) >= min_dip_pct:

                    # 2. ФИЛЬТР ВОЗДУХА: Строго 50% долины должно быть пустым
                    highs_between = highs[idx_left_peak+1 : idx_right_peak]
                    if len(highs_between) > 0:
                        if np.mean(highs_between) > (avg_peak_high - dip_depth_usd * (1 - noise_space)):
                            continue

                    # 3. МАКРО-ТРЕНД: Проверяем НАЧАЛО тренда (база из 3 свечей за 20 баров до пика)
                    pre_start = max(0, idx_left_peak - 20)
                    trend_base_avg = np.mean(closes[pre_start : pre_start+3])

                    if trend_base_avg < valley_low:

                        # 4. ТРИГГЕР: Падение на 0.75 ATR от правого пика
                        if closes[i] < highs[idx_right_peak] - (current_atr_usd * 0.75):
                            data.at[data.index[i], 'Target'] = DT
                            continue

        # ==========================================
        # DOUBLE BOTTOM (DB)
        # ==========================================
        if (min_width <= db_width[i] <= max_width) and (db_sym[i] <= atr_threshold):

            if l_idx_1[i] >= (window_size - 3):
                continue

            idx_left_valley = i - l_idx_1[i]
            idx_right_valley = i - l_idx_2[i]

            if idx_left_valley >= 0 and idx_left_valley < idx_right_valley:
                peak_high = np.max(highs[idx_left_valley : idx_right_valley])
                avg_valley_low = (lows[idx_left_valley] + lows[idx_right_valley]) / 2
                rise_height_usd = peak_high - avg_valley_low

                # 1. ЗАЩИТА ОТ ШУМА
                if rise_height_usd > (dip_atr_min * current_atr_usd) and (rise_height_usd / closes[i]) >= min_dip_pct:

                    # 2. ФИЛЬТР ВОЗДУХА
                    lows_between = lows[idx_left_valley+1 : idx_right_valley]
                    if len(lows_between) > 0:
                        if np.mean(lows_between) < (avg_valley_low + rise_height_usd * (1 - noise_space)):
                            continue

                    # 3. МАКРО-ТРЕНД: Проверяем НАЧАЛО падения
                    pre_start = max(0, idx_left_valley - 20)
                    trend_base_avg = np.mean(closes[pre_start : pre_start+3])

                    if trend_base_avg > peak_high:

                        # 4. ТРИГГЕР: Рост на 0.75 ATR от правого дна
                        if closes[i] > lows[idx_right_valley] + (current_atr_usd * 0.75):
                            data.at[data.index[i], 'Target'] = DB

    return data

In [ ]:
def apply_true_nms(df, distance_threshold=10):
    """
    УЛЬТРА-ФАСТ ВЕРСИЯ NMS НА NUMPY.
    Работает на основе вектора подавления (Boolean Mask).
    Снижает время обработки с часов до долей секунды.
    """
    data = df.copy()

    # 1. Мгновенно вытаскиваем данные в чистые массивы C-уровня (убираем оверхед .iloc)
    targets = data['Target'].values
    dt_sym = data['DT_Symmetry_Prc'].values
    db_sym = data['DB_Symmetry_Prc'].values

    # Находим индексы всех строк, где эвристика нашла хоть какой-то паттерн
    candidate_indices = np.where(targets != TARGET_MAP["NOISE"])[0]

    if len(candidate_indices) == 0:
        return data

    # 2. Векторно собираем "скоры" (симметрию) для каждого кандидата
    candidate_targets = targets[candidate_indices]
    candidate_scores = np.where(
        candidate_targets == TARGET_MAP["DT"],
        dt_sym[candidate_indices],
        db_sym[candidate_indices]
    )

    # 3. Сортируем индексы по возрастанию score (лучшая симметрия уходит в начало)
    sort_idx = np.argsort(candidate_scores)
    sorted_indices = candidate_indices[sort_idx]
    sorted_targets = candidate_targets[sort_idx]

    # 4. Создаем маску подавления (True там, где паттерн уже "задавлен" более сильным соседом)
    is_suppressed = np.zeros(len(data), dtype=bool)

    winners_indices = []
    winners_targets = []

    # 5. Жадное подавление (работает со скоростью света)
    for idx, target in zip(sorted_indices, sorted_targets):
        # Если эта свеча уже попала в радиус подавления более красивого паттерна — скипаем её
        if is_suppressed[idx]:
            continue

        # Если не подавлен — перед нами локальный геометрический пик (победитель)
        winners_indices.append(idx)
        winners_targets.append(target)

        # Векторно за секунду "выжигаем" всех соседей в радиусе вокруг победителя
        start = max(0, idx - distance_threshold)
        end = min(len(data), idx + distance_threshold + 1)
        is_suppressed[start:end] = True

    # 6. Формируем финальный очищенный вектор таргет-классов
    new_targets = np.zeros(len(data), dtype=int)
    if winners_indices:
        new_targets[winners_indices] = winners_targets

    data['Target'] = new_targets
    return data

In [ ]:
labeled_data = {}
for symbol in tqdm(symbols, desc = "Labeling candidates"):
    labeled_data[symbol] = label_dt_db_candidates(featured_data[symbol],
                                                                 window_size=WINDOW_SIZE,
                                                   atr_threshold=0.4, #symmetry
                                                   min_width=8,
                                                   max_width=30,
                                                   dip_atr_min=1.7, #depth
                                                   min_dip_pct=0.010, #depth in absolute val
                                                   noise_space=0.6 #shadow flexibility
                                                   )

In [ ]:
clear_labeled_data = {}
for symbol in tqdm(symbols, desc="Labeling Candidates"):
    clear_labeled_data[symbol] = apply_true_nms(labeled_data[symbol], distance_threshold=7)

In [ ]:
total_dt = 0
total_db = 0

print("=== СТАТИСТИКА РАЗМЕТКИ ПОСЛЕ NMS ===")
for symbol, df in clear_labeled_data.items():
    counts = df['Target'].value_counts()

    dt_count = counts.get(TARGET_MAP["DT"], 0)
    db_count = counts.get(TARGET_MAP["DB"], 0)

    total_dt += dt_count
    total_db += db_count

    print(f"[{symbol}] Double Tops: {dt_count} | Double Bottoms: {db_count}")

print("-" * 37)
print(f"ВСЕГО НАЙДЕНО -> DT: {total_dt} | DB: {total_db} | total: {total_dt + total_db}")
candles_cnt = sum(len(raw_candles[symbol]) for symbol in symbols)
print(f"Generally it covers (rude count): {(total_dt + total_db) * 50} candles ({(total_dt + total_db) * 50 / candles_cnt * 100 : .2f}%)")
print(f"The amount of candles is {candles_cnt}")

## Output potential candidates

In [ ]:
def plot_dtdb_candidates_style(labeled_dict, target_class, num_charts=3, window_size=50):
    """
    Рисует свечные графики кандидатов DT/DB в фирменном стиле (Matplotlib).
    Использует ТОЛЬКО словарь clear_labeled_data.
    """
    # 1. Собираем глобальный пул всех индексов паттернов со всех монет
    pool = []
    for symbol, df in labeled_dict.items():
        # Временно сбрасываем индекс, чтобы работать с числовыми позициями строк
        temp_df = df.reset_index()
        matches = temp_df[temp_df['Target'] == target_class].index.tolist()
        for idx in matches:
            pool.append((symbol, idx))

    if not pool:
        print(f"⚠️ Во всем словаре не найдено паттернов класса {target_class}.")
        return

    # 2. Выбираем случайные паттерны
    dynamic_seed = int(time.time() * 1000) % (2**32 - 1)
    rng = np.random.default_rng(dynamic_seed)

    # Защита от случая, если кандидатов меньше, чем num_charts
    sample_size = min(num_charts, len(pool))
    chosen = [pool[i] for i in rng.choice(len(pool), size=sample_size, replace=False)]

    class_map = {
        CLASS_INDICES['NOISE']: "Noise",
        CLASS_INDICES['DT']: "Double Top",
        CLASS_INDICES['DB']: "Double Bottom",
    }
    padding = int(window_size * 0.33)

    for symbol, pos in chosen:
        # Достаем датафрейм монеты
        df = labeled_dict[symbol].reset_index()
        y_true_val = df.loc[pos, 'Target']

        # Заглушки для сохранения стиля заголовка (пока нет модели)
        y_pred_val = y_true_val
        probas = (
            [0.11, 0.89, 0.00]
            if y_true_val == CLASS_INDICES['DT']
            else [0.05, 0.05, 0.90]
        )

        # Формируем границы среза
        start_pos = max(0, pos - window_size - padding)
        end_pos = min(len(df), pos + padding + 1)
        vis_df = df.iloc[start_pos:end_pos].copy()

        fig, ax = plt.subplots(figsize=(12, 5))
        x_vals = np.arange(len(vis_df))

        trigger_x = pos - start_pos
        pattern_start_x = trigger_x - window_size

        # Определяем колонки OHLC
        cols = [c.lower() for c in vis_df.columns]
        o_col, h_col = vis_df.columns[cols.index('open')], vis_df.columns[cols.index('high')]
        l_col, c_col = vis_df.columns[cols.index('low')], vis_df.columns[cols.index('close')]

        up = vis_df[vis_df[c_col] >= vis_df[o_col]]
        down = vis_df[vis_df[c_col] < vis_df[o_col]]

        # Зеленые свечи
        ax.vlines(x_vals[vis_df[c_col] >= vis_df[o_col]], up[l_col], up[h_col], color='#26A69A', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] >= vis_df[o_col]], up[c_col] - up[o_col], bottom=up[o_col], color='#26A69A', width=0.7)

        # Красные свечи
        ax.vlines(x_vals[vis_df[c_col] < vis_df[o_col]], down[l_col], down[h_col], color='#EF5350', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] < vis_df[o_col]], down[o_col] - down[c_col], bottom=down[c_col], color='#EF5350', width=0.7)

        trigger_y = vis_df.iloc[trigger_x][c_col]

        # 1. Красные пунктирные рамки сканирования
        ax.axvline(pattern_start_x, color="red", linestyle="--", linewidth=2, alpha=0.7)
        ax.axvline(trigger_x, color="red", linestyle="--", linewidth=2, alpha=0.7)

        # 2. Легкая заливка зоны паттерна
        bg_color = (
            "red" if y_pred_val == CLASS_INDICES['DT'] else "green"
        )
        ax.axvspan(pattern_start_x, trigger_x, color=bg_color, alpha=0.05)

        # 3. Синяя точка (момент принятия решения моделью)
        if 0 <= trigger_x < len(vis_df):
            ax.scatter(trigger_x, trigger_y, color="blue", s=100, zorder=5, edgecolors='black', label="Сигнал алгоритма")

        # Настраиваем красивые метки времени (по оси X)
        step = max(1, len(vis_df) // 10)
        ax.set_xticks(x_vals[::step])
        labels = []

        # Ищем колонку со временем
        x_col = 'Datetime' if 'Datetime' in vis_df.columns else ('timestamp' if 'timestamp' in vis_df.columns else 'index')

        for i in range(0, len(vis_df), step):
            if x_col in vis_df.columns:
                idx_val = vis_df.iloc[i][x_col]
            else:
                idx_val = vis_df.index[i]
            labels.append(idx_val.strftime('%m-%d %H:%M') if hasattr(idx_val, 'strftime') else str(idx_val))

        ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)

        # Форматируем заголовок (Точь-в-точь как на скрине)
        probas_str = "[" + ", ".join(f"{p:.2f}" for p in probas) + "]"
        title = f"REAL Candidate | Idx={pos} | {symbol}\nTrue={class_map[y_true_val]} -> Pred={class_map[y_pred_val]}\nProbs={probas_str}"

        ax.set_title(title, fontsize=11)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper left")
        plt.tight_layout()
        plt.show()

In [ ]:
# Построить 5 графиков Double Top
plot_dtdb_candidates_style(
    labeled_dict=clear_labeled_data,
    target_class=CLASS_INDICES['DT'],
    num_charts=10,
    window_size=50
)

# Построить 5 графиков Double Bottom
plot_dtdb_candidates_style(
    labeled_dict=clear_labeled_data,
    target_class=CLASS_INDICES['DB'],
    num_charts=10,
    window_size=50
)

# Functions for working with DB

In [ ]:
def upload_heuristic_sample_to_s3(labeled_dict, s3_client, bucket_name, s3_folder, target_type, sample_size=200, window_size=50, max_workers=15):
    """
    Потокобезопасная загрузка графиков в S3.
    target_type: может быть числом (например, 1) или списком (например, [1, 2]).
    sample_size: число (200) или "all" / None для выгрузки всех.
    """

    # 1. Умная обработка входного типа (превращаем в список, если передано одно число)
    if isinstance(target_type, (int, float)):
        target_types = [int(target_type)]
    else:
        target_types = list(target_type)

    # Словарь-помощник для названий и префиксов
    meta_map = {
        CLASS_INDICES['DT']: {
            "name": "Double Top", "prefix": "DT", "color": "red"
        },
        CLASS_INDICES['DB']: {
            "name": "Double Bottom", "prefix": "DB", "color": "green"
        },
    }

    types_str = " & ".join([meta_map.get(t, {}).get("name", f"Type_{t}") for t in target_types])
    print(f"☁️ Подготовка [{types_str}] к загрузке в S3 (Бакет: {bucket_name}, Папка: {s3_folder})")

    # 2. Собираем глобальный пул кандидатов (теперь сохраняем и тип таргета!)
    pool = []
    for symbol, df in labeled_dict.items():
        temp_df = df.reset_index()
        # Фильтруем сразу по нескольким таргетам через .isin()
        matches = temp_df[temp_df['Target'].isin(target_types)].index.tolist()
        for idx in matches:
            tgt = int(temp_df.loc[idx, 'Target'])
            pool.append((symbol, idx, tgt)) # <-- Добавили tgt в кортеж

    if not pool:
        print(f"⚠️ Не найдено ни одного паттерна для отправки.")
        return

    # 3. Логика выборки (Sample или ВСЕ)
    if sample_size is None or str(sample_size).lower() == "all":
        print(f"📊 Выбрана загрузка АБСОЛЮТНО ВСЕХ кандидатов ({len(pool)} шт.)...")
        chosen_samples = pool
    else:
        actual_size = min(sample_size, len(pool))
        print(f"📊 Всего найдено: {len(pool)}. Случайная выборка: {actual_size} шт...")
        dynamic_seed = int(time.time() * 1000) % (2**32 - 1)
        rng = np.random.default_rng(dynamic_seed)
        chosen_samples = [pool[i] for i in rng.choice(len(pool), size=actual_size, replace=False)]

    padding = int(window_size * 0.33)

    # 4. ВОРКЕР (Рисует и отправляет 1 картинку)
    def process_and_upload(sample):
        # Теперь мы распаковываем ТРИ значения, включая таргет
        symbol, idx, target_val = sample
        df = labeled_dict[symbol].reset_index()

        # Получаем метаданные для конкретно этого графика
        t_meta = meta_map.get(target_val, {"name": "Unknown", "prefix": "UNK", "color": "gray"})
        pattern_name = t_meta["name"]
        prefix_name = t_meta["prefix"]
        bg_color = t_meta["color"]

        start_pos = max(0, idx - window_size - padding)
        end_pos = min(len(df), idx + padding + 1)
        vis_df = df.iloc[start_pos:end_pos].copy()

        if vis_df.empty:
            return False

        fig = Figure(figsize=(12, 5))
        canvas = FigureCanvasAgg(fig)
        ax = fig.add_subplot(111)

        x_vals = np.arange(len(vis_df))
        trigger_x = idx - start_pos
        pattern_start_x = trigger_x - window_size

        cols = [c.lower() for c in vis_df.columns]
        o_col, h_col = vis_df.columns[cols.index('open')], vis_df.columns[cols.index('high')]
        l_col, c_col = vis_df.columns[cols.index('low')], vis_df.columns[cols.index('close')]

        up = vis_df[vis_df[c_col] >= vis_df[o_col]]
        down = vis_df[vis_df[c_col] < vis_df[o_col]]

        ax.vlines(x_vals[vis_df[c_col] >= vis_df[o_col]], up[l_col], up[h_col], color='#26A69A', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] >= vis_df[o_col]], up[c_col] - up[o_col], bottom=up[o_col], color='#26A69A', width=0.7)
        ax.vlines(x_vals[vis_df[c_col] < vis_df[o_col]], down[l_col], down[h_col], color='#EF5350', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] < vis_df[o_col]], down[o_col] - down[c_col], bottom=down[c_col], color='#EF5350', width=0.7)

        trigger_y = vis_df.iloc[trigger_x][c_col]

        ax.axvline(pattern_start_x, color="red", linestyle="--", linewidth=2, alpha=0.7)
        ax.axvline(trigger_x, color="red", linestyle="--", linewidth=2, alpha=0.7)

        # Динамический цвет фона
        ax.axvspan(pattern_start_x, trigger_x, color=bg_color, alpha=0.05)

        if 0 <= trigger_x < len(vis_df):
            ax.scatter(trigger_x, trigger_y, color="blue", s=100, zorder=5, edgecolors='black', label="Сигнал алгоритма")

        step = max(1, len(vis_df) // 10)
        ax.set_xticks(x_vals[::step])
        labels = []
        x_col = 'Datetime' if 'Datetime' in vis_df.columns else ('timestamp' if 'timestamp' in vis_df.columns else 'index')
        for i in range(0, len(vis_df), step):
            idx_val = vis_df.iloc[i][x_col] if x_col in vis_df.columns else vis_df.index[i]
            labels.append(idx_val.strftime('%m-%d %H:%M') if hasattr(idx_val, 'strftime') else str(idx_val))
        ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)

        # Динамический заголовок
        ax.set_title(f"Heuristic Evaluation | Idx={idx} | {symbol}\nAlgorithm Detected: {pattern_name}", fontsize=11)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper left")

        fig.tight_layout()

        img_buffer = io.BytesIO()
        fig.savefig(img_buffer, format='png', dpi=100)
        img_buffer.seek(0)

        safe_symbol = symbol.replace("/", "_")

        # Динамическое имя файла
        filename = f"{s3_folder}/{prefix_name}_idx_{idx}_{safe_symbol}.png"

        s3_client.upload_fileobj(
            img_buffer,
            bucket_name,
            filename,
            ExtraArgs={'ContentType': 'image/png'}
        )

        del fig
        del canvas

        return True

    uploaded_count = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = executor.map(process_and_upload, chosen_samples)
        for _ in tqdm(futures, total=len(chosen_samples), desc="Uploading Patterns"):
            uploaded_count += 1

    print(f"✅ Готово! Успешно загружено {uploaded_count} картинок в S3.")

In [ ]:
def clear_s3_folder_fast(s3_client, bucket_name, folder_prefix):
    """
    ПАКЕТНАЯ ОЧИСТКА: Удаляет до 1000 файлов за 1 сетевой запрос.
    Использует пагинацию для поддержки папок любого размера (даже > 1000 файлов).
    """
    if not folder_prefix.endswith('/'):
        folder_prefix += '/'

    print(f"🗑️ Начинаем очистку папки s3://{bucket_name}/{folder_prefix} ...")

    # Инициализируем пагинатор, чтобы прочитать ВСЕ файлы, сколько бы их ни было
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=folder_prefix)

    deleted_count = 0

    # Создаем "бездонный" прогресс-бар, так как заранее не знаем точное количество файлов
    with tqdm(desc="  ↳ Удаление файлов", unit=" шт.") as pbar:
        for page in pages:
            if 'Contents' not in page:
                continue

            # 1. Формируем список ключей в формате, который требует boto3
            objects_to_delete = [{'Key': obj['Key']} for obj in page['Contents']]

            # 2. Удаляем весь батч (до 1000 штук) ОДНИМ запросом
            s3_client.delete_objects(
                Bucket=bucket_name,
                Delete={
                    'Objects': objects_to_delete,
                    'Quiet': True # Quiet=True ускоряет парсинг ответа от сервера
                }
            )

            # Обновляем счетчики
            batch_size = len(objects_to_delete)
            deleted_count += batch_size
            pbar.update(batch_size)

    if deleted_count == 0:
        print("🤷‍♂️ В папке пусто, удалять нечего!")
    else:
        print(f"✅ Успешно удалено файлов: {deleted_count} (Пакетное удаление)")

# Test labeling (with DataBase)

## Pushing candidates in DB for test hand labeling on Human signal

In [ ]:
s3_client = boto3.client(
    service_name='s3',
    endpoint_url='https://storage.yandexcloud.net',
    aws_access_key_id=ACCESS_KEY_ID,
    aws_secret_access_key=SECRET_ACCESS_KEY
)

In [ ]:
# 1. Отправляем 200 Double Tops (DT)
upload_heuristic_sample_to_s3(
    labeled_dict=clear_labeled_data,
    s3_client=s3_client,
    bucket_name=YANDEX_S3_BUCKET,
    s3_folder=sample_patterns_prefix,
    target_type=TARGET_MAP["DT"],   # 1 = DT
    sample_size=200,
    max_workers=5
)

# 2. Отправляем 200 Double Bottoms (DB)
upload_heuristic_sample_to_s3(
    labeled_dict=clear_labeled_data,
    s3_client=s3_client,
    bucket_name=YANDEX_S3_BUCKET,
    s3_folder=sample_patterns_prefix,
    target_type=TARGET_MAP["DB"],   # 2 = DB
    sample_size=200,
    max_workers=5
)

## Clear S3 backet on yandex cloud

In [ ]:
clear_s3_folder_fast(s3_client, YANDEX_S3_BUCKET, sample_patterns_prefix)

## Describing results of test labeling on Human signal

Among 200 DB candidates were 87 TP cases (43.5%)
Among 200 DT candidates were 73 TP cases (36.5%)

As a consequance expected number of dataset for training would contain 2215 * 0.365 = 808 DT candidates and 2466 * 0.435 = 1072 DB candidates

# Main labeling (with DataBase)

## Pushing all candidates in DB for next labeling

In [ ]:
s3_client = boto3.client(
    service_name='s3',
    endpoint_url='https://storage.yandexcloud.net',
    aws_access_key_id=ACCESS_KEY_ID,
    aws_secret_access_key=SECRET_ACCESS_KEY
)

In [ ]:
upload_heuristic_sample_to_s3(
    labeled_dict=clear_labeled_data,
    s3_client=s3_client,
    bucket_name=YANDEX_S3_BUCKET,
    s3_folder=main_dataset_prefix,
    target_type=[TARGET_MAP["DB"], TARGET_MAP["DT"]],
    sample_size="all",
    max_workers=10
)

## Clear all candidates from DB

In [ ]:
clear_s3_folder_fast(s3_client, YANDEX_S3_BUCKET, main_dataset_prefix)

# Preparation for training


## Load source data

### Functions for creating train and validation data

In [12]:
def encode_labels(label):
    """Кодирует ручные метки Label Studio в классы DT/DB/Noise."""
    if pd.isna(label):
        return np.nan

    label_str = str(label).strip().casefold()

    if "double top" in label_str or "dt" in label_str:
        return TARGET_MAP['DT']
    if "double bottom" in label_str or "db" in label_str:
        return TARGET_MAP['DB']
    if (
        "noise" in label_str
        or "шум" in label_str
        or "флэт" in label_str
        or label_str in {"0", "0.0"}
    ):
        return TARGET_MAP['NOISE']

    return np.nan

### Load labeled picture indices and raw data

In [13]:
# Pooling labeled data from google drive
labeled_DT_DB_path = '/content/drive/MyDrive/Crypto_Labeled_Data/labeled_DT_DB.csv'

labeled_DT_DB = pd.read_csv(labeled_DT_DB_path, encoding='utf-8')
labeled_DT_DB['Target'] = labeled_DT_DB['label'].apply(encode_labels)

In [14]:
display(labeled_DT_DB.head())

,agreement,annotation_id,annotator,created_at,id,image,label,lead_time,updated_at,Target
0,100.0,98675095.0,amirgaf29@gmail.com,2026-07-03T13:53:42.204443Z,273780490,s3://tickframe-candidates/labeled_train_v1/DB_...,Шум / Флэт,63.530,2026-07-03T13:53:42.204457Z,0.0
1,100.0,98675117.0,amirgaf29@gmail.com,2026-07-03T13:53:57.970569Z,273780494,s3://tickframe-candidates/labeled_train_v1/DB_...,Double Bottom (W),14.031,2026-07-03T13:53:57.970581Z,2.0
2,100.0,98675129.0,amirgaf29@gmail.com,2026-07-03T13:54:05.605327Z,273780496,s3://tickframe-candidates/labeled_train_v1/DB_...,Double Bottom (W),5.270,2026-07-03T13:54:05.605338Z,2.0
3,100.0,98675140.0,amirgaf29@gmail.com,2026-07-03T13:54:15.261268Z,273780498,s3://tickframe-candidates/labeled_train_v1/DB_...,Double Bottom (W),8.223,2026-07-03T13:54:15.261281Z,2.0
4,100.0,98675151.0,amirgaf29@gmail.com,2026-07-03T13:54:24.408233Z,273780501,s3://tickframe-candidates/labeled_train_v1/DB_...,Шум / Флэт,7.549,2026-07-03T13:54:24.408247Z,0.0


### Load raw 5-minute candles

In [15]:
raw_input_folder = '/content/drive/MyDrive/Crypto_Raw_Data'
raw_candles = {}

print("Downloading raw data from disk")
# Добавляем tqdm для отслеживания загрузки сырых данных
for symbol in tqdm(symbols, desc="Загрузка сырых данных"):
    clean_name = symbol.replace('/', '_') + '_raw.csv'
    path = f'{raw_input_folder}/{clean_name}'
    if os.path.exists(path):
        raw_candles[symbol] = pd.read_csv(path, index_col=0, parse_dates=True)

print(f"✅ Загружено {len(raw_candles)} символов.")

Загрузка сырых данных:   0%|          | 0/23 [00:00<?, ?it/s]

✅ Загружено 23 символов.


In [16]:
display(raw_candles[symbols[0]].head())

,Open,High,Low,Close,Volume
Datetime,,,,,
2018-01-01 00:00:00,13704.00,13708.57,13679.42,13680.00,20.503984
2018-01-01 00:05:00,13679.42,13680.00,13616.54,13619.03,52.307490
2018-01-01 00:10:00,13619.03,13621.38,13593.16,13600.02,39.299696
2018-01-01 00:15:00,13599.97,13603.54,13519.66,13519.81,43.473560
2018-01-01 00:20:00,13519.66,13564.21,13501.87,13502.92,43.954285


## Merge raw candles and labeled pictures

In [17]:
def parse_s3_image_url(image_url: str):
    """Парсит URL картинки и возвращает (Symbol, Candle_Index)"""
    if pd.isna(image_url):
        return None, None

    base_name = os.path.basename(str(image_url))
    name_without_ext, _ = os.path.splitext(base_name)

    if "_idx_" not in name_without_ext:
        return None, None

    parts = name_without_ext.split("_idx_")
    right_part = parts[1]

    idx_str, safe_symbol = right_part.split('_', 1)
    idx = int(idx_str)

    if "_USDT" in safe_symbol:
        symbol = safe_symbol.replace("_USDT", "/USDT")
    else:
        symbol = safe_symbol.replace("_", "/", 1)

    return symbol, idx

In [18]:
# Предполагается, что датафрейм labeled_DT_DB уже загружен.
# Распаковываем результат парсинга в две новые колонки.
labeled_DT_DB['Symbol'], labeled_DT_DB['Candle_Index'] = zip(*labeled_DT_DB['image'].apply(parse_s3_image_url))

# Удаляем строки, где парсинг не удался (если такие есть)
labeled_DT_DB = labeled_DT_DB.dropna(subset=['Symbol', 'Candle_Index'])
# Убеждаемся, что индекс свечи - это целое число
labeled_DT_DB['Candle_Index'] = labeled_DT_DB['Candle_Index'].astype(int)

print(f"✅ Извлечено {len(labeled_DT_DB)} валидных меток.")
display(labeled_DT_DB[['image', 'Symbol', 'Candle_Index', 'Target']].head())

✅ Извлечено 4681 валидных меток.


,image,Symbol,Candle_Index,Target
0,s3://tickframe-candidates/labeled_train_v1/DB_...,AVAX/USDT,10014,0.0
1,s3://tickframe-candidates/labeled_train_v1/DB_...,AVAX/USDT,10022,2.0
2,s3://tickframe-candidates/labeled_train_v1/DB_...,AVAX/USDT,10030,2.0
3,s3://tickframe-candidates/labeled_train_v1/DB_...,AVAX/USDT,10038,2.0
4,s3://tickframe-candidates/labeled_train_v1/DB_...,ADA/USDT,10055,0.0


### Merge raw candles and labeled pictures

In [19]:
data = {}
for sym in tqdm(symbols, desc="Merging and Cleaning"):
    if sym not in raw_candles:
        continue

    df = raw_candles[sym].copy()
    df['Target'] = TARGET_MAP["NOISE"]

    sym_labels = labeled_DT_DB[labeled_DT_DB['Symbol'] == sym]

    if not sym_labels.empty:
        target_col_idx = df.columns.get_loc('Target')
        for _, row in sym_labels.iterrows():
            c_idx = row['Candle_Index']
            tgt = row['Target']
            if 0 <= c_idx < len(df):
                df.iat[c_idx, target_col_idx] = tgt

    # Удаляем строки, где Target превратился в NaN после encode_labels
    df = df.dropna(subset=['Target'])
    # Принудительно приводим к int, так как после NaN колонка может стать float
    df['Target'] = df['Target'].astype(int)

    data[sym] = df

print(f"✅ Данные очищены от -1/NaN и готовы для {len(data)} монет!")

Merging and Cleaning:   0%|          | 0/23 [00:00<?, ?it/s]

✅ Данные очищены от -1/NaN и готовы для 23 монет!


### Sanity check merged data

In [20]:
for sym, df in data.items():
    patterns = df[df['Target'] > 0]

    if not patterns.empty:
        print(f"📊 Статистика для: {sym}")
        print(f"Всего свечей: {len(df)}")
        print(
            f"Из них шума ({CLASS_INDICES['NOISE']}): "
            f"{len(df[df['Target'] == CLASS_INDICES['NOISE']])}"
        )
        print(
            f"Паттернов DT ({CLASS_INDICES['DT']}): "
            f"{len(df[df['Target'] == CLASS_INDICES['DT']])}"
        )
        print(
            f"Паттернов DB ({CLASS_INDICES['DB']}): "
            f"{len(df[df['Target'] == CLASS_INDICES['DB']])}"
        )

        print("-" * 40)
        print("Примеры свечей, где есть паттерн:")
        if sym == CONFIG.symbols[0]:
            display(patterns.head())

📊 Статистика для: BTC/USDT
Всего свечей: 888066
Из них шума (0): 888051
Паттернов DT (1): 4
Паттернов DB (2): 11
----------------------------------------
Примеры свечей, где есть паттерн:


,Open,High,Low,Close,Volume,Target
Datetime,,,,,,
2018-05-23 18:30:00,7519.60,7540.33,7515.80,7532.49,44.896100,1
2018-05-23 19:10:00,7509.92,7532.19,7508.46,7531.37,96.721800,1
2018-11-29 00:20:00,4260.98,4260.98,4239.94,4249.39,127.581351,2
2018-11-29 01:10:00,4244.28,4250.61,4232.11,4235.08,94.760419,2
2018-12-19 03:15:00,3733.66,3740.00,3711.90,3720.20,279.936972,1


📊 Статистика для: ETH/USDT
Всего свечей: 888061
Из них шума (0): 888011
Паттернов DT (1): 21
Паттернов DB (2): 29
----------------------------------------
Примеры свечей, где есть паттерн:
📊 Статистика для: SOL/USDT
Всего свечей: 577029
Из них шума (0): 576980
Паттернов DT (1): 28
Паттернов DB (2): 21
----------------------------------------
Примеры свечей, где есть паттерн:
📊 Статистика для: BNB/USDT
Всего свечей: 473350
Из них шума (0): 473324
Паттернов DT (1): 16
Паттернов DB (2): 10
----------------------------------------
Примеры свечей, где есть паттерн:
📊 Статистика для: XRP/USDT
Всего свечей: 888071
Из них шума (0): 888030
Паттернов DT (1): 17
Паттернов DB (2): 24
----------------------------------------
Примеры свечей, где есть паттерн:
📊 Статистика для: ADA/USDT
Всего свечей: 853513
Из них шума (0): 853429
Паттернов DT (1): 44
Паттернов DB (2): 40
----------------------------------------
Примеры свечей, где есть паттерн:
📊 Статистика для: DOT/USDT
Всего свечей: 602953
Из них 

In [21]:
# Собираем общую статистику по всем символам
stats_list = []

for sym, df in data.items():
    counts = df['Target'].value_counts()
    stats_list.append({
        'Symbol': sym,
        'Total Candles': len(df),
        f"Noise ({CLASS_INDICES['NOISE']})": counts.get(CLASS_INDICES['NOISE'], 0),
        f"DT ({CLASS_INDICES['DT']})": counts.get(CLASS_INDICES['DT'], 0),
        f"DB ({CLASS_INDICES['DB']})": counts.get(CLASS_INDICES['DB'], 0)
    })

# Создаем DataFrame для удобного отображения
summary_stats = pd.DataFrame(stats_list)

# Добавляем строку 'Total' в конец
total_row = summary_stats.select_dtypes(include=[np.number]).sum()
total_summary = pd.concat([summary_stats, pd.DataFrame([{'Symbol': 'TOTAL', **total_row}])], ignore_index=True)

display(total_summary)

,Symbol,Total Candles,Noise (0),DT (1),DB (2)
0,BTC/USDT,888066,888051,4,11
1,ETH/USDT,888061,888011,21,29
2,SOL/USDT,577029,576980,28,21
3,BNB/USDT,473350,473324,16,10
4,XRP/USDT,888071,888030,17,24
5,ADA/USDT,853513,853429,44,40
6,DOT/USDT,602953,602867,45,41
7,LINK/USDT,784396,784302,57,37
8,AVAX/USDT,594317,594270,17,30
9,DOGE/USDT,749836,749796,16,24


## Save labeled data

In [22]:
import os
from tqdm.auto import tqdm

# Путь к новой папке на Google Drive
output_folder = '/content/drive/MyDrive/Crypto_Merged_Labeled_Data'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"✅ Папка создана: {output_folder}")

print("💾 Сохранение размеченных данных...")
# Добавляем tqdm для отслеживания процесса сохранения
for sym, df in tqdm(data.items(), desc="Сохранение монет"):
    clean_name = sym.replace('/', '_') + '_labeled.csv'
    save_path = os.path.join(output_folder, clean_name)
    df.to_csv(save_path)

print(f"✅ Успешно сохранено {len(data)} файлов.")

💾 Сохранение размеченных данных...


Сохранение монет:   0%|          | 0/23 [00:00<?, ?it/s]

✅ Успешно сохранено 23 файлов.


Подготовка обучающего dataset продолжается этапами загрузки, feature transformation, batch storage и split.

## Storage loading


### Load one labeled symbol

In [14]:
# 1. Хранилище исходных labeled-файлов остаётся на Google Drive.
LABELED_DATA_DIR = Path('/content/drive/MyDrive/Crypto_Merged_Labeled_Data')

# 2. Batch-файлы создаются только во временной локальной файловой системе runtime.
COLAB_BATCH_ROOT = Path('/content/tickframe_batches')
LOCAL_BATCH_ROOT = Path(tempfile.gettempdir()) / 'tickframe_batches'
BATCH_ROOT = COLAB_BATCH_ROOT if COLAB_BATCH_ROOT.parent.exists() else LOCAL_BATCH_ROOT

# 3. Манифест и chunks живут в одной dedicated-директории.
BATCH_MANIFEST_PATH = BATCH_ROOT / 'manifest.json'

# Persistent copy used by the training-only load block.
BATCH_DRIVE_ROOT = Path('/content/drive/MyDrive/tickframe_batches')
BATCH_DRIVE_MANIFEST_PATH = BATCH_DRIVE_ROOT / 'manifest.json'
BATCH_CHUNK_ROWS = 100_000
MIN_FREE_BYTES = 2 * 1024**3

# 4. Итоговую модель можно сохранять отдельно от временных dataset chunks.
MODEL_DIR = Path('/content/drive/MyDrive/Crypto_pattern_detection_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def reset_batch_root(root):
    """Очищает только dedicated batch-каталог перед новым построением dataset."""
    root = Path(root)
    if root.exists():
        for child in root.iterdir():
            if child.is_dir():
                shutil.rmtree(child)
            else:
                child.unlink()
    root.mkdir(parents=True, exist_ok=True)


def ensure_batch_disk_space(root, required_bytes=MIN_FREE_BYTES):
    """Останавливает pipeline до записи, если локальный диск почти заполнен."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    free_bytes = shutil.disk_usage(root).free
    if free_bytes < required_bytes:
        raise RuntimeError(
            f'Недостаточно места для chunks: свободно {free_bytes / 2**30:.2f} GiB, '
            f'требуется минимум {required_bytes / 2**30:.2f} GiB.'
        )

## Feature and label transformation


### Feature engineering functions

In [24]:
@njit
def _find_dt_db_extrema_numba(high_windows, low_windows, window_size, min_dist):
    """
    Ускоренный поиск 2-х пиков и 2-х впадин для DT/DB.
    """
    n_windows = len(high_windows)
    macro_h_idx = np.zeros((n_windows, 2))
    macro_h_prc = np.zeros((n_windows, 2))
    macro_l_idx = np.zeros((n_windows, 2))
    macro_l_prc = np.zeros((n_windows, 2))

    for i in range(n_windows):
        h_win = high_windows[i]
        l_win = low_windows[i]

        # Пики
        avail_h = np.ones(window_size, dtype=np.bool_)
        for s in range(2):
            best_val = -1e10
            best_idx = -1
            for j in range(window_size):
                if avail_h[j] and h_win[j] > best_val:
                    best_val = h_win[j]
                    best_idx = j
            if best_idx != -1:
                macro_h_idx[i, s] = window_size - best_idx
                macro_h_prc[i, s] = best_val
                avail_h[max(0, best_idx - min_dist) : min(window_size, best_idx + min_dist + 1)] = False

        # Впадины
        avail_l = np.ones(window_size, dtype=np.bool_)
        for s in range(2):
            best_val = 1e10
            best_idx = -1
            for j in range(window_size):
                if avail_l[j] and l_win[j] < best_val:
                    best_val = l_win[j]
                    best_idx = j
            if best_idx != -1:
                macro_l_idx[i, s] = window_size - best_idx
                macro_l_prc[i, s] = best_val
                avail_l[max(0, best_idx - min_dist) : min(window_size, best_idx + min_dist + 1)] = False

    return macro_h_idx, macro_h_prc, macro_l_idx, macro_l_prc

# Реализация add_smart_features перенесена в раннюю секцию
# `## Add smart features to candles` для последовательного запуска notebook.

### Applying smart features

## Dataset batch classes


### DatasetBatchManifest

In [15]:
@dataclass
class DatasetBatchRecord:
    """Описание одного физического chunk-файла без загрузки его массивов."""

    split: str
    symbol: str
    group_id: int
    chunk_id: int
    path: str
    rows: int
    row_start: int
    row_end: int
    feature_dtype: str
    label_dtype: str
    group_dtype: str


@dataclass
class DatasetBatchManifest:
    """Маленький JSON-контракт между writer, loader и training pipeline."""

    root: str
    feature_columns: list
    chunk_rows: int
    records: list
    version: str = '1'

    def save(self, path):
        """Сохраняет только metadata; сами X/y массивы остаются в chunks."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            'version': self.version,
            'root': self.root,
            'feature_columns': self.feature_columns,
            'chunk_rows': self.chunk_rows,
            'records': self.records,
        }
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

    @classmethod
    def load(cls, path):
        """Восстанавливает manifest без чтения dataset arrays в RAM."""
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        # Resolve the root from the manifest location, not from the old runtime.
        # This makes a copied /content manifest usable from Google Drive.
        manifest_root = Path(path).parent
        return cls(
            root=str(manifest_root),
            feature_columns=payload['feature_columns'],
            chunk_rows=payload['chunk_rows'],
            records=payload['records'],
            version=payload.get('version', '1'),
        )

    def records_for(self, split, symbol=None):
        """Возвращает metadata в стабильном порядке split → symbol → chunk."""
        records = [
            record for record in self.records
            if record['split'] == split
            and (symbol is None or record['symbol'] == symbol)
        ]
        # Preserve writer insertion order: original symbols order, then chunk order.
        return records

    def symbols_for(self, split):
        """Возвращает уникальные symbols, не загружая ни одного chunk."""
        return list(dict.fromkeys(record['symbol'] for record in self.records_for(split)))

### DatasetBatchWriter

In [16]:
class DatasetBatchWriter:
    """Пишет chronological split одного symbol в локальные joblib chunks."""

    def __init__(
        self,
        root,
        feature_columns,
        chunk_rows=BATCH_CHUNK_ROWS,
        train_fraction=CONFIG.train_fraction,
        calibration_fraction_of_holdout=CONFIG.calibration_fraction_of_holdout,
    ):
        # Каталог writer всегда является dedicated runtime cache.
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.feature_columns = list(feature_columns)
        self.chunk_rows = int(chunk_rows)
        self.train_fraction = train_fraction
        self.calibration_fraction_of_holdout = calibration_fraction_of_holdout
        self.records = []

    def _validate_frame(self, frame, symbol):
        """Проверяет schema до выделения NumPy-копий текущего symbol."""
        missing = sorted(set(self.feature_columns) - set(frame.columns))
        if missing:
            raise ValueError(f'{symbol}: отсутствуют признаки {missing}')
        if 'Target' not in frame.columns:
            raise ValueError(f'{symbol}: отсутствует колонка Target')

    def _write_chunk(self, split, symbol, group_id, chunk_id, frame, row_start):
        """Конвертирует только один chunk и сразу сериализует его на диск."""
        # Выборка признаков создаёт одну bounded NumPy-копию текущего chunk.
        x_values = frame.loc[:, self.feature_columns].to_numpy(copy=True)
        # Target и group ids хранятся отдельно, чтобы loader мог читать их независимо.
        y_values = frame['Target'].to_numpy(copy=True)
        group_dtype = (
            np.int16
            if group_id <= np.iinfo(np.int16).max
            else np.int32
        )
        groups = np.full(len(y_values), group_id, dtype=group_dtype)

        # Каталог содержит split и symbol, поэтому имена файлов не конфликтуют.
        symbol_dir = self.root / split / symbol.replace('/', '_')
        symbol_dir.mkdir(parents=True, exist_ok=True)
        chunk_path = symbol_dir / f'chunk_{chunk_id:06d}.joblib'
        joblib.dump(
            {'X': x_values, 'y': y_values, 'groups': groups},
            chunk_path,
            compress=3,
        )

        # Metadata не содержит массивов и безопасен для JSON manifest.
        self.records.append(
            {
                'split': split,
                'symbol': symbol,
                'group_id': int(group_id),
                'chunk_id': int(chunk_id),
                'path': str(chunk_path.relative_to(self.root)),
                'rows': int(len(y_values)),
                'row_start': int(row_start),
                'row_end': int(row_start + len(y_values)),
                'feature_dtype': str(x_values.dtype),
                'label_dtype': str(y_values.dtype),
                'group_dtype': str(groups.dtype),
            }
        )

        # Удаляем текущие массивы до перехода к следующему chunk.
        del x_values, y_values, groups
        gc.collect()

    def write_symbol(self, frame, symbol, group_id):
        """Пишет все три split текущего symbol и не сохраняет frame внутри writer."""
        self._validate_frame(frame, symbol)
        frame = frame.sort_index() if not frame.index.is_monotonic_increasing else frame

        train_end = int(len(frame) * self.train_fraction)
        calibration_end = train_end + int(
            (len(frame) - train_end) * self.calibration_fraction_of_holdout
        )
        split_ranges = {
            'train': (0, train_end),
            'calibration': (train_end, calibration_end),
            'validation': (calibration_end, len(frame)),
        }

        for split_name, (split_start, split_end) in split_ranges.items():
            # Только один bounded split view существует в этой итерации.
            split_frame = frame.iloc[split_start:split_end]
            for chunk_id, chunk_start in enumerate(
                range(0, len(split_frame), self.chunk_rows)
            ):
                chunk_frame = split_frame.iloc[chunk_start:chunk_start + self.chunk_rows]
                self._write_chunk(
                    split_name,
                    symbol,
                    group_id,
                    chunk_id,
                    chunk_frame,
                    row_start=chunk_start,
                )
                del chunk_frame
            del split_frame
            gc.collect()

        del frame
        gc.collect()

    def finalize(self):
        """Создаёт manifest после завершения записи всех symbols."""
        manifest = DatasetBatchManifest(
            root=str(self.root),
            feature_columns=self.feature_columns,
            chunk_rows=self.chunk_rows,
            records=self.records,
        )
        manifest.save(self.root / 'manifest.json')
        return manifest

### DatasetBatchLoader and DatasetPatternSampler

In [ ]:
class DatasetBatchLoader:
    """Итерирует chunks и освобождает payload перед следующей загрузкой."""

    def __init__(self, manifest, feature_columns=None):
        self.manifest = manifest
        self.root = Path(manifest.root)
        manifest_columns = list(manifest.feature_columns)
        if feature_columns is None:
            self.feature_indices = None
        else:
            missing = [
                column for column in feature_columns
                if column not in manifest_columns
            ]
            if missing:
                raise ValueError(
                    f'В manifest отсутствуют feature columns: {missing}'
                )
            self.feature_indices = np.asarray(
                [manifest_columns.index(column) for column in feature_columns],
                dtype=int,
            )

    def _select_features(self, X):
        if self.feature_indices is None:
            return X
        if hasattr(X, 'iloc'):
            return X.iloc[:, self.feature_indices]
        return np.asarray(X)[:, self.feature_indices]

    @staticmethod
    def _take_rows(X, indices):
        if hasattr(X, 'iloc'):
            return X.iloc[indices]
        return X[indices]

    def iter_batches(self, split, symbol=None):
        """Yield одного chunk payload за раз; полный split в RAM не собирается."""
        for record in self.manifest.records_for(split, symbol):
            chunk_path = Path(record['path'])
            if not chunk_path.is_absolute():
                chunk_path = self.root / chunk_path
            payload = joblib.load(chunk_path)
            yield record, payload
            del payload
            gc.collect()

    def load_symbol_labels(self, split, symbol):
        """Загружает только labels одного symbol для sampling индексов."""
        y_parts = []
        for _, payload in self.iter_batches(split, symbol):
            y_parts.append(np.asarray(payload['y']))
            del payload
        return np.concatenate(y_parts) if y_parts else np.array([], dtype=np.int8)

    def load_sampled_train(self, noise_percent, dead_zone, random_state=RANDOM_STATE):
        """Загружает только bounded train sample, а не весь train dataset."""
        x_parts, y_parts, group_parts = [], [], []
        rng = np.random.default_rng(random_state)

        for symbol in tqdm(
            self.manifest.symbols_for('train'),
            desc='Сбор bounded train sample из chunks',
        ):
            labels = self.load_symbol_labels('train', symbol)
            pattern_indices = np.flatnonzero(labels != TARGET_MAP['NOISE'])
            protected = np.zeros(len(labels), dtype=bool)
            for pattern_index in pattern_indices:
                left = max(0, pattern_index - dead_zone)
                right = min(len(labels), pattern_index + dead_zone + 1)
                protected[left:right] = True

            clear_noise = np.flatnonzero(
                (labels == TARGET_MAP['NOISE']) & ~protected
            )
            required_noise = int(
                len(pattern_indices) * noise_percent / (1.0 - noise_percent)
            )
            n_pick = min(required_noise, len(clear_noise))
            noise_indices = (
                rng.choice(clear_noise, size=n_pick, replace=False)
                if n_pick
                else np.array([], dtype=int)
            )
            selected = np.sort(np.concatenate((pattern_indices, noise_indices)))

            # Читаем только выбранные rows из каждого chunk текущего symbol.
            for record, payload in self.iter_batches('train', symbol):
                left = record['row_start']
                right = record['row_end']
                local_indices = selected[(selected >= left) & (selected < right)] - left
                if len(local_indices):
                    x_chunk = self._take_rows(payload['X'], local_indices)
                    x_parts.append(self._select_features(x_chunk))
                    y_parts.append(payload['y'][local_indices])
                    group_parts.append(payload['groups'][local_indices])
                del payload

            del labels, protected, clear_noise, selected
            gc.collect()

        if not x_parts:
            raise ValueError('После sampling не осталось train rows.')
        return (
            np.concatenate(x_parts, axis=0),
            np.concatenate(y_parts, axis=0),
            np.concatenate(group_parts, axis=0),
        )

    def predict_split(self, model, split):
        """Считает probabilities по symbols с известным total без X в RAM."""
        y_parts, probability_parts, group_parts = [], [], []
        split_symbols = self.manifest.symbols_for(split)
        for symbol in tqdm(
            split_symbols,
            total=len(split_symbols),
            desc=f'Предсказания по symbols: {split}',
        ):
            for _, payload in self.iter_batches(split, symbol):
                X_chunk = self._select_features(payload['X'])
                y_parts.append(np.asarray(payload['y']))
                group_parts.append(np.asarray(payload['groups']))
                probability_parts.append(model.predict_proba(X_chunk))
                del X_chunk, payload
        return (
            np.concatenate(y_parts),
            np.concatenate(probability_parts, axis=0),
            np.concatenate(group_parts),
        )


class DatasetPatternSampler:
    """Документирует bounded sampling как отдельный batch-aware слой."""

    def __init__(self, noise_percent, dead_zone, random_state=RANDOM_STATE):
        self.noise_percent = noise_percent
        self.dead_zone = dead_zone
        self.random_state = random_state

    def build_train_arrays(self, loader):
        """Делегирует запись bounded sample loader-у, не меняя sampling contract."""
        return loader.load_sampled_train(
            noise_percent=self.noise_percent,
            dead_zone=self.dead_zone,
            random_state=self.random_state,
        )


## Build train, calibration and validation batches


### Write per-symbol batches

In [28]:
# 1. Каждый запуск строит dedicated chunks заново в локальном runtime storage.
reset_batch_root(BATCH_ROOT)
ensure_batch_disk_space(BATCH_ROOT)

# 2. Writer не получает глобальный data dict: он получает один symbol за раз.
batch_writer = DatasetBatchWriter(
    root=BATCH_ROOT,
    feature_columns=FEATURE_COLUMNS_NAME,
    chunk_rows=BATCH_CHUNK_ROWS,
)

available_symbols = []
for group_id, sym in enumerate(
    tqdm(symbols, desc='Загрузка, feature engineering и запись chunks')
):
    # 3. Загружаем только один labeled CSV с Google Drive.
    clean_name = sym.replace('/', '_') + '_labeled.csv'
    source_path = LABELED_DATA_DIR / clean_name
    if not source_path.exists():
        print(f'⚠️ Файл для {sym} не найден: {source_path}')
        continue

    frame = pd.read_csv(source_path, index_col=0, parse_dates=True)
    # 4. Feature engineering выполняется только для текущего symbol.
    frame = add_smart_features(frame, window_size=WINDOW_SIZE)
    batch_writer.write_symbol(frame, sym, group_id)
    available_symbols.append(sym)

    # 5. Освобождаем pandas DataFrame до чтения следующего symbol.
    del frame
    gc.collect()

batch_manifest = batch_writer.finalize()
print(f'✅ Chunks записаны: {len(batch_manifest.records)} файлов')
print(f'✅ Manifest: {BATCH_MANIFEST_PATH}')

Загрузка, feature engineering и запись chunks:   0%|          | 0/23 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

...

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Chunks записаны: 163 файлов
✅ Manifest: /content/tickframe_batches/manifest.json


### Finalize manifest

In [29]:
# 1. Загружаем только metadata и не материализуем arrays.
batch_manifest = DatasetBatchManifest.load(BATCH_MANIFEST_PATH)
batch_loader = DatasetBatchLoader(batch_manifest)

# 2. Проверяем feature contract и количество chunks каждого split.
if list(batch_manifest.feature_columns) != list(FEATURE_COLUMNS_NAME):
    raise ValueError('Manifest feature schema не совпадает с FEATURE_COLUMNS_NAME')

for split_name in ('train', 'calibration', 'validation'):
    rows = sum(
        record['rows']
        for record in batch_manifest.records_for(split_name)
    )
    chunks = len(batch_manifest.records_for(split_name))
    print(f'{split_name}: rows={rows}, chunks={chunks}')

train: rows=10581112, chunks=117
calibration: rows=1322639, chunks=23
validation: rows=1322650, chunks=23


## Save batch chunks to Google Drive

In [30]:
# 1. Проверяем, что локальный batch dataset действительно построен.
if not BATCH_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Локальный manifest не найден: {BATCH_MANIFEST_PATH}. '
        'Сначала выполни Build train, calibration and validation batches.'
    )

# 2. Создаём persistent каталог на Google Drive.
BATCH_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# 3. Копируем chunks с прогрессом; manifest записываем отдельно.
# Это также мигрирует старый manifest с абсолютных /content-путей на относительные.
batch_files = [
    file_path
    for file_path in BATCH_ROOT.rglob('*')
    if file_path.is_file() and file_path.name != 'manifest.json'
]
for file_path in tqdm(batch_files, desc='Сохранение batch chunks в Google Drive'):
    relative_path = file_path.relative_to(BATCH_ROOT)
    destination = BATCH_DRIVE_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file_path, destination)

manifest_payload = json.loads(BATCH_MANIFEST_PATH.read_text(encoding='utf-8'))
for record in manifest_payload['records']:
    record_path = Path(record['path'])
    if record_path.is_absolute():
        record['path'] = str(record_path.relative_to(BATCH_ROOT))
BATCH_DRIVE_MANIFEST_PATH.write_text(
    json.dumps(manifest_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

# 4. Проверяем, что persistent manifest появился.
if not BATCH_DRIVE_MANIFEST_PATH.exists():
    raise RuntimeError(
        f'Копирование batch dataset не завершилось: {BATCH_DRIVE_MANIFEST_PATH}'
    )
print(f'✅ Batch dataset сохранён в: {BATCH_DRIVE_ROOT}')


Сохранение batch chunks в Google Drive:   0%|          | 0/163 [00:00<?, ?it/s]

✅ Batch dataset сохранён в: /content/drive/MyDrive/tickframe_batches


## Verify local batch manifest

### Check local chunk counts

In [31]:
# Локальный manifest проверяется только как результат Build-блока.
# Training использует отдельный manifest из Google Drive.
if not BATCH_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Локальный manifest не найден: {BATCH_MANIFEST_PATH}')

local_manifest = DatasetBatchManifest.load(BATCH_MANIFEST_PATH)
print(f'✅ Локальный manifest готов: {len(local_manifest.records)} chunks')

✅ Локальный manifest готов: 163 chunks


# Training model pipeline

## Load batch data from Google Drive

In [18]:
# 1. Загружаем только manifest из persistent Drive-каталога.
if not BATCH_DRIVE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Batch manifest не найден: {BATCH_DRIVE_MANIFEST_PATH}. '
        'Сначала сохрани chunks через блок Save batch chunks to Google Drive.'
    )

batch_manifest = DatasetBatchManifest.load(BATCH_DRIVE_MANIFEST_PATH)
batch_loader = DatasetBatchLoader(batch_manifest)

# 2. Проверяем feature contract до sampling и training.
if list(batch_manifest.feature_columns) != list(FEATURE_COLUMNS_NAME):
    raise ValueError(
        'Feature schema в Drive manifest не совпадает с FEATURE_COLUMNS_NAME'
    )

# 3. Загружаем только bounded train sample; все исходные chunks остаются на Drive.
train_sampler = DatasetPatternSampler(
    noise_percent=CONFIG.noise_percent,
    dead_zone=CONFIG.dead_zone,
    random_state=RANDOM_STATE,
)
X_train, y_train, TRAIN_GROUPS = train_sampler.build_train_arrays(batch_loader)
print(f'✅ Batch data загружены из Drive: X={X_train.shape}, y={y_train.shape}')


Сбор bounded train sample из chunks:   0%|          | 0/23 [00:00<?, ?it/s]

✅ Batch data загружены из Drive: X=(2417, 18), y=(2417,)


## Functions for training

In [19]:
def _group_ranges(groups):
    """Возвращает непрерывные [start, end) диапазоны одного символа."""
    groups = np.asarray(groups)
    if len(groups) == 0:
        return []
    boundaries = np.flatnonzero(groups[1:] != groups[:-1]) + 1
    starts = np.r_[0, boundaries]
    ends = np.r_[boundaries, len(groups)]
    return list(zip(starts, ends))

def apply_label_tolerance(y, tolerance=5, groups=None):
    """Расширяет метки только внутри одного символа, не через его границу."""
    if tolerance <= 0:
        return y.copy() if hasattr(y, 'copy') else np.asarray(y).copy()

    y_values = np.asarray(y).copy()
    result = y_values.copy()
    group_values = np.asarray(groups) if groups is not None else np.zeros(len(y_values), dtype=int)

    for start, end in _group_ranges(group_values):
        for class_idx in (TARGET_MAP['DT'], TARGET_MAP['DB']):
            indices = np.where(y_values[start:end] == class_idx)[0] + start
            for idx in indices:
                left = max(start, idx - tolerance)
                right = min(end, idx + tolerance + 1)
                result[left:right] = class_idx

    return pd.Series(result, index=y.index) if isinstance(y, pd.Series) else result

In [20]:
def find_best_threshold_fbeta(y_true, probas, class_idx, beta=2.5):
    """
    Находит оптимальный порог вероятности для максимизации F-beta score.
    """
    y_true_binary = (y_true == class_idx).astype(int)
    y_scores = probas[:, class_idx]

    precisions, recalls, thresholds = precision_recall_curve(y_true_binary, y_scores)

    beta_sq = beta ** 2
    # Избегаем деления на ноль
    denominator = (beta_sq * precisions) + recalls
    fbeta_scores = np.divide((1 + beta_sq) * (precisions * recalls),
                             denominator,
                             out=np.zeros_like(denominator),
                             where=denominator != 0)

    best_idx = int(np.argmax(fbeta_scores))
    if len(thresholds) == 0:
        return 1.0, fbeta_scores[best_idx], recalls[best_idx], precisions[best_idx]

    best_thresh = thresholds[min(best_idx, len(thresholds) - 1)]
    return best_thresh, fbeta_scores[best_idx], recalls[best_idx], precisions[best_idx]

In [ ]:
class DatasetPattern:
    """Групповой dataset-слой для DT/DB train и validation."""
    def __init__(
        self,
        X,
        y,
        groups,
        training=True,
        noise_percent=0.90,
        dead_zone=80,
        shuffle=False,
        random_state=42,
    ):
        self.X = X
        self.y = np.asarray(y).astype(int)
        self.groups = np.asarray(groups)
        self.training = training
        self.dead_zone = dead_zone if training else 0
        self.random_state = random_state
        rng = np.random.default_rng(random_state)
        selected = []

        for start, end in _group_ranges(self.groups):
            local_y = self.y[start:end]
            pattern_indices = np.flatnonzero(local_y != TARGET_MAP['NOISE']) + start
            protected = np.zeros(end - start, dtype=bool)
            for idx in pattern_indices - start:
                left = max(0, idx - self.dead_zone)
                right = min(end - start, idx + self.dead_zone + 1)
                protected[left:right] = True

            clear_noise = np.flatnonzero(
                (local_y == TARGET_MAP['NOISE']) & ~protected
            ) + start

            if training:
                required_noise = int(
                    len(pattern_indices) * noise_percent / (1.0 - noise_percent)
                )
                n_pick = min(required_noise, len(clear_noise))
                noise_indices = rng.choice(
                    clear_noise, size=n_pick, replace=False
                ) if n_pick else np.array([], dtype=int)
            else:
                # Validation/calibration сохраняют весь noise: это боевая ситуация.
                noise_indices = clear_noise

            selected.extend(pattern_indices.tolist())
            selected.extend(np.asarray(noise_indices, dtype=int).tolist())

        self.final_indices = np.asarray(selected, dtype=int)
        if shuffle:
            rng.shuffle(self.final_indices)

    def __len__(self):
        return len(self.final_indices)

    def __getitem__(self, index):
        original_index = self.final_indices[index]
        if hasattr(self.X, 'iloc'):
            return self.X.iloc[original_index], self.y[original_index]
        return self.X[original_index], self.y[original_index]


def auto_tune_xgboost_fbeta(X_train, y_train, n_iter=15, tolerance=5, beta=2.5, target_device=ACTIVE_DEVICE, train_is_sampled=False, train_groups=None):
    """
    Тюнинг XGBoost на полном датасете.
    ОПТИМИЗАЦИЯ: Добавлен вложенный tqdm для фолдов кросс-валидации.
    """
    print(f"⌒ Применение толерантности ({tolerance} свечей) к разметке...")
    groups_for_training = (
        np.asarray(TRAIN_GROUPS)
        if train_groups is None
        else np.asarray(train_groups)
    )
    y_train_tolerant = apply_label_tolerance(
        y_train,
        tolerance=tolerance,
        groups=groups_for_training,
    )

    param_grid = {
        'n_estimators': [200, 300, 400, 500],
        'learning_rate': [0.01, 0.05],
        'max_depth': [3, 4, 5],
        'gamma': [0, 1, 3],
        'max_delta_step': [1, 2],
        'reg_alpha': [0, 0.1, 1.0],
        'reg_lambda': [1.0, 3.0],
        'subsample': [0.8],
        'colsample_bytree': [0.8],
        'min_child_weight': [2, 5, 10]
    }

    tscv = TimeSeriesSplit(n_splits=3)
    best_global_score = -1.0
    best_params = None

    print(f"⚙℘ Оптимизация F{beta} на {target_device.upper()}")
    renew_random_state()
    rng = np.random.default_rng(RANDOM_STATE)
    if train_is_sampled:
        # DatasetBatchLoader уже выполнил bounded dead-zone/noise sampling.
        X_fit_values = np.asarray(X_train)
        y_fit_values = np.asarray(y_train_tolerant)
    else:
        # Старый in-memory путь сохраняется для обратной совместимости.
        train_dataset = DatasetPattern(
            X_train,
            y_train_tolerant,
            groups_for_training,
            training=True,
            noise_percent=CONFIG.noise_percent,
            dead_zone=CONFIG.dead_zone,
            shuffle=False,
            random_state=RANDOM_STATE,
        )
        sample_idx = train_dataset.final_indices
        X_fit_values = X_train[sample_idx]
        y_fit_values = y_train_tolerant[sample_idx]
    sample_weights = compute_sample_weight(
        class_weight='balanced',
        y=y_fit_values,
    )
    required_classes = np.asarray(
        sorted(CLASS_INDICES.values()),
        dtype=int,
    )
    cv_splits = []
    for split_number, (train_idx, val_idx) in enumerate(
        tscv.split(X_fit_values),
        start=1,
    ):
        train_classes = np.unique(y_fit_values[train_idx])
        validation_classes = np.unique(y_fit_values[val_idx])
        missing_train = sorted(
            set(required_classes) - set(train_classes)
        )
        missing_validation = sorted(
            {
                CLASS_INDICES['DT'],
                CLASS_INDICES['DB'],
            } - set(validation_classes)
        )
        if missing_train or missing_validation:
            print(
                f'⚠️ Пропуск CV fold {split_number}: '
                f'missing train={missing_train}, '
                f'missing validation patterns={missing_validation}'
            )
            continue
        cv_splits.append((train_idx, val_idx))

    if not cv_splits:
        raise RuntimeError(
            'TimeSeriesSplit не создал ни одного валидного fold: '
            'в train/validation fold отсутствуют необходимые классы.'
        )
    print(
        f"📉 Обучающая DatasetPattern: {len(X_fit_values)} строк; "
        f"валидных CV folds: {len(cv_splits)}"
    )

    for i in tqdm(range(n_iter), desc="Tuning Iterations"):
        # Выбираем значение ровно один раз для каждого параметра.
        params = {}
        for key, values in param_grid.items():
            value = rng.choice(values)
            params[key] = value.item() if hasattr(value, 'item') else value
        fold_scores = []

        # Добавлен tqdm для фолдов
        for f_idx, (train_idx, val_idx) in enumerate(tqdm(cv_splits, desc=f"  ↳ Fold Progress (Iter {i+1})", leave=False)):
            # Используем заранее подготовленные массивы и fold boundaries.
            X_f_train = X_fit_values[train_idx]
            y_f_train = y_fit_values[train_idx]
            w_f_train = sample_weights[train_idx]

            X_f_val = X_fit_values[val_idx]
            y_f_val = y_fit_values[val_idx]

            if len(y_f_train) == 0 or len(y_f_val) == 0:
                continue

            model = xgb.XGBClassifier(
                **params,
                tree_method='hist',
                device=target_device,
                objective='multi:softprob',
                num_class=3,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=0
            )

            model.fit(
                X_f_train, y_f_train,
                sample_weight=w_f_train,
                eval_set=[(X_f_val, y_f_val)],
                verbose=False
            )

            probas = model.predict_proba(X_f_val)
            _, f_dt, _, _ = find_best_threshold_fbeta(y_f_val, probas, 1, beta=beta)
            _, f_db, _, _ = find_best_threshold_fbeta(y_f_val, probas, 2, beta=beta)
            fold_scores.append((f_dt + f_db) / 2)

        if not fold_scores: continue
        avg_fbeta = np.mean(fold_scores)
        if avg_fbeta > best_global_score:
            best_global_score = avg_fbeta
            best_params = params

        # ДОБАВЛЕНО ЗДЕСЬ: Вывод параметров и метрики после завершения всех фолдов в текущей итерации
        tqdm.write(f"Iter {i+1} | Avg F{beta}: {avg_fbeta:.4f} | Best: {best_global_score:.4f}")
        tqdm.write(f"  ↳ Params: {params}\n")

    # Финальное обучение на той же bounded выборке, что использовалась при tuning.
    y_final = y_fit_values
    final_classes = np.unique(y_final)
    if not np.array_equal(final_classes, required_classes):
        raise RuntimeError(
            f'Финальный train sample не содержит все классы: '
            f'получены {final_classes}, ожидались {required_classes}.'
        )

    if best_params is None:
        raise RuntimeError(
            'Tuning не вернул параметры модели: отсутствуют valid fold scores'
        )

    print(f"\n✅ Финальное обучение на {len(y_final)} примерах...")
    final_model = xgb.XGBClassifier(
        **best_params,
        tree_method='hist',
        device=target_device,
        objective='multi:softprob',
        num_class=3,
        random_state=RANDOM_STATE
    )
    final_model.fit(X_fit_values, y_final, sample_weight=sample_weights, verbose=True)

    return final_model, best_params


In [22]:
def multiclass_macro_pr_auc(y_true, y_score_probas, num_classes=3):
    """
    Macro PR-AUC (average_precision) по всем классам.
    Оценивает общее качество разделения классов моделью.
    """
    y_true = np.asarray(y_true)
    scores = []
    for cl in range(num_classes):
        # Бинаризуем метки для текущего класса
        y_true_binary = (y_true == cl).astype(int)
        # Берем столбец вероятностей для этого класса
        y_scores = y_score_probas[:, cl]
        scores.append(average_precision_score(y_true_binary, y_scores))

    return float(np.mean(scores)) if scores else 0.0

In [23]:
def get_tolerant_indices(y_true, y_pred, class_idx, tolerance=5, groups=None):
    """One-to-one TP/FP/FN matching inside each symbol group."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    group_values = np.asarray(groups) if groups is not None else np.zeros(len(y_true), dtype=int)
    tp_list, fp_list, fn_list = [], [], []

    for start, end in _group_ranges(group_values):
        true_idxs = np.where(y_true[start:end] == class_idx)[0] + start
        pred_idxs = np.where(y_pred[start:end] == class_idx)[0] + start
        matched_truths = set()

        for p in pred_idxs:
            candidates = [
                t for t in true_idxs
                if t not in matched_truths and abs(t - p) <= tolerance
            ]
            if not candidates:
                fp_list.append(p)
                continue
            matched = min(candidates, key=lambda t: abs(t - p))
            matched_truths.add(matched)
            tp_list.append(p)

        fn_list.extend(t for t in true_idxs if t not in matched_truths)

    return np.asarray(tp_list), np.asarray(fp_list), np.asarray(fn_list)


def evaluate_with_tolerance(
    y_true,
    y_pred,
    tolerance=5,
    classes=(CLASS_INDICES['DT'], CLASS_INDICES['DB']),
    groups=None,
):
    """Единая one-to-one оценка с tolerance и границами символов."""
    results = {}
    class_names = {
        CLASS_INDICES['DT']: 'Double Top (DT)',
        CLASS_INDICES['DB']: 'Double Bottom (DB)',
    }

    for cls in classes:
        tp, fp, fn = get_tolerant_indices(
            y_true, y_pred, class_idx=cls, tolerance=tolerance, groups=groups
        )
        precision = len(tp) / (len(tp) + len(fp)) if len(tp) + len(fp) else 0
        recall = len(tp) / (len(tp) + len(fn)) if len(tp) + len(fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        results[class_names[cls]] = {
            'Precision': round(precision, 4),
            'Recall': round(recall, 4),
            'F1 Score': round(f1, 4),
            'TP': len(tp), 'FP': len(fp), 'FN': len(fn),
        }

    return pd.DataFrame(results).T

In [24]:
def plot_model_evaluation_curves(y_true, y_probas):
    """
    Визуализация Precision-Recall кривых для классов DT и DB.
    Помогает выбрать порог уверенности для торговой стратегии.
    """
    plt.figure(figsize=(15, 6))

    colors = {1: '#EF5350', 2: '#26A69A'}
    names = {1: 'Double Top (DT)', 2: 'Double Bottom (DB)'}

    for cls in [1, 2]:
        y_true_binary = (y_true == cls).astype(int)
        precision, recall, thresholds = precision_recall_curve(y_true_binary, y_probas[:, cls])
        pr_auc = auc(recall, precision)

        plt.plot(recall, precision, color=colors[cls], lw=2,
                 label=f'{names[cls]} (AUC = {pr_auc:.4f})')

    plt.xlabel('Recall (Полнота)')
    plt.ylabel('Precision (Точность)')
    plt.title('Multi-class Precision-Recall Curves')
    plt.legend(loc='best')
    plt.grid(alpha=0.3)
    plt.show()

## Train XGBoost model

In [25]:
# Запуск процесса обучения на bounded train sample.
beta_val = 2
tolerance_val = CONFIG.tolerance

model, best_cfg = auto_tune_xgboost_fbeta(
    X_train=X_train,
    y_train=y_train,
    n_iter=20,
    tolerance=tolerance_val,
    beta=beta_val,
    train_is_sampled=True,
)

# Модель — небольшой итоговый артефакт, поэтому сохраняем её отдельно.
model_save_path = MODEL_DIR / 'xgb_dtdb_model.json'
model.save_model(str(model_save_path))
print(f'\n✅ Модель обучена и сохранена в {model_save_path}')

⌒ Применение толерантности (10 свечей) к разметке...
⚙℘ Оптимизация F2 на CUDA
📉 Обучающая DatasetPattern: 2417 строк


Tuning Iterations:   0%|          | 0/20 [00:00<?, ?it/s]

  ↳ Fold Progress (Iter 1):   0%|          | 0/3 [00:00<?, ?it/s]

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got [1 2]

In [ ]:
# Обучение уже запускается в предыдущей ячейке.
# Эта ячейка оставлена как явная точка продолжения notebook pipeline.
print('✅ Training cell 109 завершила подготовку model pipeline.')

# Evaluating model pipeline

## Functions for evaluating model

In [ ]:
def find_threshold_for_target_recall(y_true, probas, class_idx, target_recall=0.70):
    """Ищет порог, при котором Recall будет максимально близок к заданному таргету."""
    y_true_binary = (y_true == class_idx).astype(int)
    y_scores = probas[:, class_idx]

    precisions, recalls, thresholds = precision_recall_curve(y_true_binary, y_scores)
    valid_idx = np.where(recalls >= target_recall)[0]

    if len(thresholds) == 0:
        # В выборке нет положительных примеров или все scores одинаковы.
        return 1.0, recalls[-1], precisions[-1]

    if len(valid_idx) == 0:
        print(f"⚠️ Внимание: Невозможно достичь Recall {target_recall} для класса {class_idx}. Берем минимум.")
        best_idx = 0
    else:
        best_idx = min(valid_idx[-1], len(thresholds) - 1)

    return thresholds[best_idx], recalls[best_idx], precisions[best_idx]

In [ ]:
def apply_pattern_thresholds(probas, threshold_dt=0.75, threshold_db=0.8):
    """Применяет пороги confidence для классов DT и DB."""
    dt_idx = CLASS_INDICES['DT']
    db_idx = CLASS_INDICES['DB']
    prob_dt, prob_db = probas[:, dt_idx], probas[:, db_idx]
    y_pred = np.zeros(len(probas), dtype=int)
    y_pred[(prob_dt >= threshold_dt) & (prob_dt > prob_db)] = dt_idx
    y_pred[(prob_db >= threshold_db) & (prob_db > prob_dt)] = db_idx
    return y_pred

In [ ]:
def apply_nms_clustering(y_pred, probas, window_size=30, groups=None):
    """Схлопывает сигналы в один лучший, не объединяя разные символы."""
    y_pred = np.asarray(y_pred).astype(int)
    y_clean = np.zeros_like(y_pred)
    group_values = np.asarray(groups) if groups is not None else np.zeros(len(y_pred), dtype=int)

    for start, end in _group_ranges(group_values):
        for cls in (TARGET_MAP['DT'], TARGET_MAP['DB']):
            idxs = np.where(y_pred[start:end] == cls)[0] + start
            k = 0
            while k < len(idxs):
                cluster = [idxs[k]]
                k += 1
                while k < len(idxs) and (idxs[k] - idxs[k - 1]) <= window_size:
                    cluster.append(idxs[k])
                    k += 1

                best_idx = cluster[int(np.argmax(probas[cluster, cls]))]
                y_clean[best_idx] = cls

    return y_clean

In [ ]:
def count_physical_patterns(y_true, class_idx, groups=None):
    """Считает физические паттерны отдельно внутри каждого символа."""
    y_true = np.asarray(y_true)
    group_values = np.asarray(groups) if groups is not None else np.zeros(len(y_true), dtype=int)
    total = 0

    for start, end in _group_ranges(group_values):
        idxs = np.where(y_true[start:end] == class_idx)[0]
        if len(idxs):
            total += int(np.sum(np.diff(idxs) > 1) + 1)

    return total

In [ ]:
def depict_feature_importance(model, feature_names=None, top_n=20):
    print("\n📊 Топ признаков по важности (Gain):")
    importance = np.asarray(model.feature_importances_)
    if feature_names is None:
        feature_names = list(getattr(model, 'feature_names_in_', []))
    feature_names = list(feature_names)
    if len(feature_names) != len(importance):
        raise ValueError(
            f"Количество имён признаков ({len(feature_names)}) не совпадает "
            f"с важностями модели ({len(importance)}). Передайте X.columns."
        )
    df_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
    df_imp = df_imp.sort_values(by='Importance', ascending=False).head(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(df_imp['Feature'][::-1], df_imp['Importance'][::-1], color='dodgerblue')
    plt.title("Топ-20 самых важных признаков (XGBoost Feature Importance)")
    plt.xlabel("Importance (Gain)")
    plt.grid(axis='x', alpha=0.3)
    plt.show()

In [ ]:
def plot_metrics_vs_confidence(
    y_true,
    probas,
    class_indices=(CLASS_INDICES['DT'], CLASS_INDICES['DB']),
    class_names=None,
    n_points=101,
):
    """Показывает Recall и Precision при изменении confidence threshold."""
    y_true = np.asarray(y_true).reshape(-1)
    probas = np.asarray(probas)
    class_names = class_names or {
        CLASS_INDICES['DT']: "Double Top (DT)",
        CLASS_INDICES['DB']: "Double Bottom (DB)",
    }
    confidence_grid = np.linspace(0.0, 1.0, n_points)
    curves = {}

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True)

    for class_idx in class_indices:
        y_binary = (y_true == class_idx).astype(int)
        class_scores = probas[:, class_idx]
        recalls, precisions = [], []

        for confidence in confidence_grid:
            predicted_positive = class_scores >= confidence
            true_positive = np.sum(predicted_positive & (y_binary == 1))
            false_positive = np.sum(predicted_positive & (y_binary == 0))
            false_negative = np.sum(~predicted_positive & (y_binary == 1))

            recall = (
                true_positive / (true_positive + false_negative)
                if true_positive + false_negative else 0.0
            )
            precision = (
                true_positive / (true_positive + false_positive)
                if true_positive + false_positive else 0.0
            )
            recalls.append(recall)
            precisions.append(precision)

        label = class_names.get(class_idx, f"Class {class_idx}")
        curves[class_idx] = {
            "confidence": confidence_grid,
            "recall": np.asarray(recalls),
            "precision": np.asarray(precisions),
        }
        axes[0].plot(confidence_grid, recalls, label=label, linewidth=2)
        axes[1].plot(confidence_grid, precisions, label=label, linewidth=2)

    axes[0].set_title("Recall vs confidence")
    axes[0].set_ylabel("Recall")
    axes[1].set_title("Precision vs confidence")
    axes[1].set_ylabel("Precision")

    for axis in axes:
        axis.set_xlabel("Confidence threshold")
        axis.set_xlim(0.0, 1.0)
        axis.set_ylim(0.0, 1.05)
        axis.grid(True, alpha=0.3)
        axis.legend()

    fig.suptitle("DT/DB quality metrics by confidence threshold")
    plt.tight_layout()
    plt.show()
    return curves

In [ ]:
%matplotlib inline
def depict_pr_auc_custom_debug(y_true, probas, threshold_dt=0.75, threshold_db=0.8):
    print("🛠 [DEBUG] 1. Вход в функцию отрисовки...")

    try:
        y_true_arr = np.array(y_true).flatten()
        print(f"🛠 [DEBUG] 2. Размерность y_true: {y_true_arr.shape}")
        print(f"🛠 [DEBUG] 3. Размерность probas: {probas.shape}")

        plt.figure(figsize=(12, 8))

        classes_to_plot = {
            CLASS_INDICES['DT']: {
                'name': 'Double Top (Шорт)',
                'color': 'crimson',
                'thresh': threshold_dt,
            },
            CLASS_INDICES['DB']: {
                'name': 'Double Bottom (Лонг)',
                'color': 'dodgerblue',
                'thresh': threshold_db,
            },
        }

        for cl, cfg in classes_to_plot.items():
            print(f"🛠 [DEBUG] 4. Обработка класса {cl}...")
            y_bin = (y_true_arr == cl).astype(int)
            prob_cls = probas[:, cl]

            precision, recall, thresholds_curve = precision_recall_curve(y_bin, prob_cls)
            pr_auc = average_precision_score(y_bin, prob_cls)
            print(
                f"🛠 [DEBUG] 5. Класс {cl}: PR-AUC = {pr_auc:.6g}, "
                f"точек на кривой: {len(precision)}"
            )

            # Не передаём миллионы точек в matplotlib: AP считается по полной
            # кривой, а для отображения используем равномерную выборку.
            max_plot_points = 5000
            plot_idx = np.linspace(
                0, len(precision) - 1,
                min(max_plot_points, len(precision)),
                dtype=int,
            )
            plt.plot(
                recall[plot_idx], precision[plot_idx], color=cfg['color'], lw=2.5,
                label=f"{cfg['name']} (PR-AUC = {pr_auc:.6g})",
            )

            if len(thresholds_curve):
                idx = int(np.argmin(np.abs(thresholds_curve - cfg['thresh'])))
                idx = min(idx, len(precision) - 1, len(recall) - 1)
                plt.plot(recall[idx], precision[idx], marker='o', markersize=10,
                         color=cfg['color'], markeredgecolor='white', markeredgewidth=2, zorder=5)
                plt.annotate(
                    f"Thresh: {cfg['thresh']:.2f}",
                    (recall[idx], precision[idx]), textcoords="offset points", xytext=(15, 10),
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=cfg['color'], alpha=0.9)
                )

        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.legend(loc='lower left')

        print("🛠 [DEBUG] 6. Команды для графика сформированы. Вызываю plt.show()...")
        print("⬇️⬇️⬇️ ГРАФИК ДОЛЖЕН БЫТЬ ПРЯМО ПОД ЭТИМ ТЕКСТОМ ⬇️⬇️⬇️")

        plt.show()

    except Exception as e:
        print(f"❌ [ОШИБКА ВНУТРИ ФУНКЦИИ]: {e}")

In [ ]:
def print_dist(y_values, title='DATASET'):
    """Печатает распределение классов без зависимости от training cells."""
    labels = np.asarray(y_values).reshape(-1)
    class_indices = globals().get(
        'CLASS_INDICES',
        {'NOISE': 0, 'DT': 1, 'DB': 2},
    )
    total = len(labels)

    print(f'--- {title} ---')
    for class_name, class_idx in class_indices.items():
        count = int(np.sum(labels == class_idx))
        share = count / total * 100 if total else 0.0
        print(f'{class_name}: {count:,} ({share:.2f}%)')
    print(f'TOTAL: {total:,}')


def load_evaluation_batch_context(manifest_path):
    """Создаёт минимальный read-only loader для standalone evaluation."""
    manifest_path = Path(manifest_path)
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    root = manifest_path.parent

    class EvaluationManifest:
        def __init__(self, root, feature_columns, records):
            self.root = Path(root)
            self.feature_columns = feature_columns
            self.records = records

        def records_for(self, split, symbol=None):
            return [
                record for record in self.records
                if record['split'] == split
                and (symbol is None or record['symbol'] == symbol)
            ]

        def symbols_for(self, split):
            return list(dict.fromkeys(
                record['symbol']
                for record in self.records_for(split)
            ))

    class EvaluationBatchLoader:
        def __init__(self, manifest):
            self.manifest = manifest

        def iter_batches(self, split, symbol=None):
            for record in self.manifest.records_for(split, symbol):
                chunk_path = Path(record['path'])
                if not chunk_path.is_absolute():
                    chunk_path = self.manifest.root / chunk_path
                payload = joblib.load(chunk_path)
                yield record, payload
                del payload

        def predict_split(self, model, split):
            y_parts, probability_parts, group_parts = [], [], []
            split_symbols = self.manifest.symbols_for(split)
            for symbol in tqdm(
                split_symbols,
                total=len(split_symbols),
                desc=f'Предсказания по symbols: {split}',
            ):
                for _, payload in self.iter_batches(split, symbol):
                    y_parts.append(np.asarray(payload['y']))
                    group_parts.append(np.asarray(payload['groups']))
                    probability_parts.append(
                        model.predict_proba(payload['X'])
                    )
                    del payload

            return (
                np.concatenate(y_parts),
                np.concatenate(probability_parts, axis=0),
                np.concatenate(group_parts),
            )

    manifest = EvaluationManifest(
        root=root,
        feature_columns=payload['feature_columns'],
        records=payload['records'],
    )
    return manifest, EvaluationBatchLoader(manifest)


def print_confusion_report(class_name, tp, fp, fn, total_samples):
    """Печатает TP/FP/FN/TN в той же логике, что и SWP pipeline."""
    tp = int(tp)
    fp = int(fp)
    fn = int(fn)
    tn = max(0, int(total_samples) - (tp + fp + fn))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    print(f'\nCLASSIFICATION REPORT: {class_name}')
    print(f'  True Positives (TP):  {tp:,}')
    print(f'  False Positives (FP): {fp:,}')
    print(f'  False Negatives (FN): {fn:,}')
    print(f'  True Negatives (TN):  {tn:,}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1 Score:  {f1:.4f}')
    return tn


## Event position logic functions

In [ ]:
# Размер одного event-level окна в свечах.
# WINDOW_SIZE равен 50, поэтому baseline EVENT_GAP=35 составляет 70% окна.
EVENT_GAP = 35


def _build_event_clusters(
    positions,
    groups,
    event_gap,
    confidence=None,
):
    """
    Группирует позиции в события отдельно внутри каждого symbol.

    positions:
        Глобальные индексы prediction или true label в validation-массивах.
    groups:
        Массив group_id такой же длины, как y_valid.
    event_gap:
        Максимальная дистанция от anchor первой позиции до остальных
        позиций того же event.
    confidence:
        Confidence соответствующего класса для prediction. Для true events
        confidence не передаётся, потому что ground truth не имеет score.

    Возвращает список словарей. Каждый словарь описывает один event:
        group_id      — symbol-группа;
        anchor        — первая позиция event;
        representative — позиция с максимальной confidence;
        members       — все позиции, вошедшие в event.
    """
    positions = np.asarray(positions, dtype=int)
    groups = np.asarray(groups)

    if event_gap < 0:
        raise ValueError('event_gap должен быть неотрицательным.')
    if len(positions) == 0:
        return []

    if confidence is not None:
        confidence = np.asarray(confidence)
        if len(confidence) != len(groups):
            raise ValueError('confidence и groups должны иметь одинаковую длину.')

    events = []
    # Обрабатываем каждый symbol отдельно: индексы разных монет нельзя
    # объединять даже если их числовая дистанция мала.
    for group_id in np.unique(groups[positions]):
        group_positions = np.sort(
            positions[groups[positions] == group_id]
        )
        current_members = []
        anchor = None

        for position in group_positions:
            position = int(position)

            # Первый сигнал становится anchor нового event.
            if anchor is None:
                anchor = position
                current_members = [position]
                continue

            # Важно: сравниваем с anchor, а не с предыдущей позицией.
            # Поэтому цепочка 100, 130, 160 при gap=35 разделится на
            # event [100, 130] и event [160], а не сольётся целиком.
            if position - anchor <= event_gap:
                current_members.append(position)
                continue

            events.append(
                _finalize_event_cluster(
                    current_members,
                    group_id,
                    confidence,
                )
            )
            anchor = position
            current_members = [position]

        if current_members:
            events.append(
                _finalize_event_cluster(
                    current_members,
                    group_id,
                    confidence,
                )
            )

    return events


def _finalize_event_cluster(members, group_id, confidence):
    """Фиксирует event и выбирает его representative-позицию."""
    members = [int(member) for member in members]

    if confidence is None:
        representative = members[0]
    else:
        # Из нескольких сигналов одного event выбираем самый уверенный.
        representative = max(
            members,
            key=lambda member: float(confidence[member]),
        )

    return {
        'group_id': int(group_id),
        'anchor': members[0],
        'representative': int(representative),
        'members': members,
    }


def _build_true_events(y_true, class_idx, groups):
    """
    Строит ground-truth events из непрерывных positive label-позиций.

    Здесь EVENT_GAP не используется. Истинные labels объединяются только
    если каждая следующая positive-позиция непосредственно продолжает
    предыдущую: 100, 101, 102 — один true event.
    Это сохраняет семантику существующего count_physical_patterns().
    """
    y_true = np.asarray(y_true)
    groups = np.asarray(groups)
    true_positions = np.flatnonzero(y_true == class_idx)
    if len(true_positions) == 0:
        return []

    true_events = []
    for group_id in np.unique(groups[true_positions]):
        group_positions = np.sort(
            true_positions[groups[true_positions] == group_id]
        )
        current_members = [int(group_positions[0])]

        for position in group_positions[1:]:
            position = int(position)
            if position == current_members[-1] + 1:
                current_members.append(position)
                continue

            true_events.append(
                _finalize_event_cluster(
                    current_members,
                    group_id,
                    confidence=None,
                )
            )
            current_members = [position]

        true_events.append(
            _finalize_event_cluster(
                current_members,
                group_id,
                confidence=None,
            )
        )

    return true_events


def evaluate_event_level(
    y_true,
    y_pred,
    probas,
    class_idx,
    groups,
    tolerance,
    event_gap=EVENT_GAP,
):
    """
    Считает event-level TP/FP/FN для одного класса.

    Порядок обработки:
    1. Берём только predictions нужного класса.
    2. Объединяем близкие predictions в events через EVENT_GAP.
    3. Оставляем representative с максимальной confidence.
    4. Строим true events из соседних ground-truth labels.
    5. Сопоставляем prediction events и true events one-to-one
       внутри одного symbol и в пределах tolerance.

    Event-level TN намеренно не вычисляется: у события нет естественного
    пространства всех отрицательных возможностей, аналогичного свечам.
    Row-level TN остаётся в основном evaluation report.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    probas = np.asarray(probas)
    groups = np.asarray(groups)

    if not (len(y_true) == len(y_pred) == len(probas) == len(groups)):
        raise ValueError('y_true, y_pred, probas и groups должны быть согласованы.')
    if tolerance < 0:
        raise ValueError('tolerance должен быть неотрицательным.')

    class_confidence = probas[:, class_idx]
    predicted_positions = np.flatnonzero(y_pred == class_idx)
    predicted_events = _build_event_clusters(
        positions=predicted_positions,
        groups=groups,
        event_gap=event_gap,
        confidence=class_confidence,
    )
    true_events = _build_true_events(y_true, class_idx, groups)

    matched_true_indices = set()
    true_positives = []
    false_positives = []

    # Сортировка делает результат deterministic независимо от порядка
    # group_id в manifest.
    predicted_events = sorted(
        predicted_events,
        key=lambda event: (event['group_id'], event['representative']),
    )

    for predicted_event in predicted_events:
        candidates = []
        for true_index, true_event in enumerate(true_events):
            if true_index in matched_true_indices:
                continue
            if true_event['group_id'] != predicted_event['group_id']:
                continue

            # Сопоставляем events по ближайшим member-позициям, а не только
            # по confidence representative. Иначе prediction [95, 105, 130]
            # с representative=130 не совпадёт с truth [100, 101, 102],
            # хотя members 95 и 105 находятся внутри tolerance.
            member_distance = min(
                abs(predicted_member - true_member)
                for predicted_member in predicted_event['members']
                for true_member in true_event['members']
            )
            if member_distance <= tolerance:
                candidates.append((true_index, true_event, member_distance))

        if not candidates:
            false_positives.append(predicted_event)
            continue

        matched_true_index, matched_true_event, _ = min(
            candidates,
            key=lambda item: item[2],
        )
        matched_true_indices.add(matched_true_index)
        true_positives.append(
            {
                'prediction': predicted_event,
                'truth': matched_true_event,
            }
        )

    false_negatives = [
        true_event
        for true_index, true_event in enumerate(true_events)
        if true_index not in matched_true_indices
    ]

    tp_count = len(true_positives)
    fp_count = len(false_positives)
    fn_count = len(false_negatives)
    precision = (
        tp_count / (tp_count + fp_count)
        if tp_count + fp_count
        else 0.0
    )
    recall = (
        tp_count / (tp_count + fn_count)
        if tp_count + fn_count
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )
    # F2 придаёт Recall больший вес, чем Precision:
    # F2 = 5 * P * R / (4 * P + R).
    f2 = (
        5 * precision * recall / (4 * precision + recall)
        if 4 * precision + recall
        else 0.0
    )

    return {
        'class_idx': int(class_idx),
        'event_gap': int(event_gap),
        'tolerance': int(tolerance),
        'predicted_events': predicted_events,
        'true_events': true_events,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'F2 Score': f2,
    }


def print_event_level_report(result, class_name):
    """Печатает event-level результат рядом с row-level отчётом."""
    print(f'\\nEVENT-LEVEL REPORT: {class_name}')
    print(f"  EVENT_GAP: {result['event_gap']} candles")
    print(f"  TP events: {result['TP']:,}")
    print(f"  FP events: {result['FP']:,}")
    print(f"  FN events: {result['FN']:,}")
    print(f"  Precision: {result['Precision']:.4f}")
    print(f"  Recall:    {result['Recall']:.4f}")
    print(f"  F1 Score:  {result['F1 Score']:.4f}")
    print(f"  F2 Score:  {result['F2 Score']:.4f}")


## Dataset distribution audit

In [ ]:
def audit_manifest_distribution(manifest):
    """
    Проверяет распределение rows и паттернов во всех batch split.

    Audit читает только payload['y'] из joblib chunks. Массивы payload['X']
    не используются и удаляются сразу после чтения labels, поэтому audit
    не превращается в дополнительную загрузку feature dataset в RAM.
    """
    split_names = ('train', 'calibration', 'validation')
    class_indices = {
        'NOISE': CLASS_INDICES['NOISE'],
        'DT': CLASS_INDICES['DT'],
        'DB': CLASS_INDICES['DB'],
    }
    audit = {}

    def count_contiguous_events(labels, class_idx):
        """Считает физические events по contiguous positive labels."""
        positions = np.flatnonzero(labels == class_idx)
        if len(positions) == 0:
            return 0
        return int(1 + np.count_nonzero(np.diff(positions) > 1))

    def load_symbol_labels(split, symbol):
        """Загружает labels одного symbol в chronological order."""
        label_parts = []
        for record in manifest.records_for(split, symbol):
            chunk_path = Path(record['path'])
            if not chunk_path.is_absolute():
                chunk_path = Path(manifest.root) / chunk_path

            payload = joblib.load(chunk_path)
            label_parts.append(np.asarray(payload['y']))
            # X не используется audit-ом; освобождаем payload сразу.
            del payload

        if not label_parts:
            return np.array([], dtype=np.int8)
        return np.concatenate(label_parts, axis=0)

    for split in split_names:
        split_symbols = (
            manifest.symbols_for(split)
            if hasattr(manifest, 'symbols_for')
            else list(dict.fromkeys(
                record['symbol']
                for record in manifest.records_for(split)
            ))
        )
        configured_symbols = getattr(globals().get('CONFIG'), 'symbols', ())
        total_symbols = len(configured_symbols) or len(split_symbols)
        split_audit = {
            'rows': 0,
            'NOISE': 0,
            'DT': 0,
            'DB': 0,
            'DT_events': 0,
            'DB_events': 0,
            'symbols': {},
        }

        for symbol in tqdm(
            split_symbols,
            total=total_symbols,
            desc=f'Audit labels по монетам: {split}',
        ):
            labels = load_symbol_labels(split, symbol)
            symbol_stats = {
                'rows': int(len(labels)),
                'NOISE': int(np.sum(labels == class_indices['NOISE'])),
                'DT': int(np.sum(labels == class_indices['DT'])),
                'DB': int(np.sum(labels == class_indices['DB'])),
                'DT_events': count_contiguous_events(
                    labels,
                    class_indices['DT'],
                ),
                'DB_events': count_contiguous_events(
                    labels,
                    class_indices['DB'],
                ),
            }
            split_audit['symbols'][symbol] = symbol_stats

            for key in ('rows', 'NOISE', 'DT', 'DB', 'DT_events', 'DB_events'):
                split_audit[key] += symbol_stats[key]
            del labels

        audit[split] = split_audit

        print(f'\n=== DATASET AUDIT: {split.upper()} ===')
        print(
            f"rows={split_audit['rows']:,} | "
            f"noise={split_audit['NOISE']:,} | "
            f"DT={split_audit['DT']:,} | DB={split_audit['DB']:,}"
        )
        print(
            f"physical DT events={split_audit['DT_events']:,} | "
            f"physical DB events={split_audit['DB_events']:,}"
        )
        for symbol, stats in split_audit['symbols'].items():
            print(
                f"  {symbol}: rows={stats['rows']:,}, "
                f"DT={stats['DT']:,} ({stats['DT_events']} events), "
                f"DB={stats['DB']:,} ({stats['DB_events']} events)"
            )

    total_rows = sum(audit[split]['rows'] for split in split_names)
    print('\n=== DATASET AUDIT: SPLIT SHARE ===')
    for split in split_names:
        share = (
            audit[split]['rows'] / total_rows * 100
            if total_rows
            else 0.0
        )
        print(
            f"{split}: {audit[split]['rows']:,} rows "
            f"({share:.2f}% of all rows)"
        )

    return audit


DATASET_AUDIT = audit_manifest_distribution(batch_manifest)


## Calibration threshold diagnostics

In [ ]:
def _threshold_candidates(
    y_true,
    probas,
    class_idx,
    raw_threshold,
    max_quantile_candidates=41,
):
    """Формирует компактный набор threshold без перебора всех rows."""
    scores = np.asarray(probas[:, class_idx], dtype=float)
    scores = scores[np.isfinite(scores)]
    if len(scores) == 0:
        return np.array([1.0], dtype=float)

    quantiles = np.linspace(0.0, 1.0, max_quantile_candidates)
    candidates = [
        np.quantile(scores, quantiles),
        np.asarray([0.0, 1.0, raw_threshold], dtype=float),
    ]
    positive_scores = np.asarray(
        probas[np.asarray(y_true) == class_idx, class_idx],
        dtype=float,
    )
    if len(positive_scores):
        candidates.append(positive_scores)

    values = np.unique(np.concatenate(candidates))
    return values[np.isfinite(values) & (values >= 0.0) & (values <= 1.0)]


def _evaluate_postprocessed_threshold(
    y_true,
    probas,
    groups,
    class_idx,
    threshold,
    nms_window,
    tolerance,
):
    """Оценивает один threshold после threshold, NMS и event matching."""
    if class_idx == CLASS_INDICES['DT']:
        threshold_dt, threshold_db = threshold, 1.1
    else:
        threshold_dt, threshold_db = 1.1, threshold

    y_pred_raw = apply_pattern_thresholds(
        probas,
        threshold_dt=threshold_dt,
        threshold_db=threshold_db,
    )
    y_pred = apply_nms_clustering(
        y_pred_raw,
        probas,
        window_size=nms_window,
        groups=groups,
    )
    tp, fp, fn = get_tolerant_indices(
        y_true,
        y_pred,
        class_idx=class_idx,
        tolerance=tolerance,
        groups=groups,
    )
    event = evaluate_event_level(
        y_true=y_true,
        y_pred=y_pred,
        probas=probas,
        class_idx=class_idx,
        groups=groups,
        tolerance=tolerance,
        event_gap=EVENT_GAP,
    )
    return {
        'threshold': float(threshold),
        'row_TP': int(len(tp)),
        'row_FP': int(len(fp)),
        'row_FN': int(len(fn)),
        'event_TP': int(event['TP']),
        'event_FP': int(event['FP']),
        'event_FN': int(event['FN']),
        'event_precision': float(event['Precision']),
        'event_recall': float(event['Recall']),
        'event_f1': float(event['F1 Score']),
        'event_f2': float(event['F2 Score']),
    }


def select_postprocessed_threshold(
    y_true,
    probas,
    groups,
    class_idx,
    target_recall,
    nms_window,
    tolerance,
):
    """
    Подбирает threshold по calibration после полного post-processing.

    Приоритет objective:
    1. event Recall >= target_recall;
    2. минимум event FP;
    3. максимум Recall и более высокий threshold при равенстве.
    Если target недостижим, возвращается лучший доступный Recall
    с явным статусом target_unmet.
    """
    raw_threshold, raw_recall, raw_precision = (
        find_threshold_for_target_recall(
            y_true,
            probas,
            class_idx=class_idx,
            target_recall=target_recall,
        )
    )
    candidates = _threshold_candidates(
        y_true,
        probas,
        class_idx,
        raw_threshold=raw_threshold,
    )
    records = [
        _evaluate_postprocessed_threshold(
            y_true=y_true,
            probas=probas,
            groups=groups,
            class_idx=class_idx,
            threshold=threshold,
            nms_window=nms_window,
            tolerance=tolerance,
        )
        for threshold in tqdm(
            candidates,
            desc=f'Threshold diagnostics class={class_idx}',
            leave=False,
        )
    ]
    feasible = [
        record for record in records
        if record['event_recall'] >= target_recall
    ]
    if feasible:
        selected = min(
            feasible,
            key=lambda record: (
                record['event_FP'],
                -record['event_recall'],
                -record['threshold'],
            ),
        )
        status = 'target_met'
    else:
        selected = max(
            records,
            key=lambda record: (
                record['event_recall'],
                -record['event_FP'],
                record['threshold'],
            ),
        )
        status = 'target_unmet'

    selected = dict(selected)
    selected.update(
        {
            'class_idx': int(class_idx),
            'target_recall': float(target_recall),
            'status': status,
            'raw_threshold': float(raw_threshold),
            'raw_recall': float(raw_recall),
            'raw_precision': float(raw_precision),
            'candidates_evaluated': int(len(records)),
            'frontier': records,
        }
    )
    return selected


## Run evaluation

In [ ]:
# Evaluation может запускаться отдельно после загрузки model и batch context.
if 'print_dist' not in globals():
    def print_dist(y_values, title='DATASET'):
        labels = np.asarray(y_values).reshape(-1)
        class_indices = globals().get(
            'CLASS_INDICES',
            {'NOISE': 0, 'DT': 1, 'DB': 2},
        )
        total = len(labels)
        print(f'--- {title} ---')
        for class_name, class_idx in class_indices.items():
            count = int(np.sum(labels == class_idx))
            share = count / total * 100 if total else 0.0
            print(f'{class_name}: {count:,} ({share:.2f}%)')
        print(f'TOTAL: {total:,}')

TARGET_RECALL = CONFIG.target_recall
TOLERANCE = CONFIG.tolerance
NMS_WINDOW = CONFIG.nms_window

# 1. Получаем calibration и validation probabilities чанками.
y_calibration, y_calibration_probas, CALIBRATION_GROUPS = (
    batch_loader.predict_split(model, 'calibration')
)
y_valid, y_valid_probas, VALID_GROUPS = (
    batch_loader.predict_split(model, 'validation')
)

validation_rows = sum(
    record['rows']
    for record in batch_manifest.records_for('validation')
)
assert len(y_valid) == validation_rows
print('📊 Validation оставлен в боевом дисбалансе:')
print_dist(y_valid, 'FULL VALIDATION SET')

# 2. Threshold подбирается только на calibration после NMS/tolerance/events.
threshold_dt_result = select_postprocessed_threshold(
    y_true=y_calibration,
    probas=y_calibration_probas,
    groups=CALIBRATION_GROUPS,
    class_idx=CLASS_INDICES['DT'],
    target_recall=TARGET_RECALL,
    nms_window=NMS_WINDOW,
    tolerance=TOLERANCE,
)
threshold_db_result = select_postprocessed_threshold(
    y_true=y_calibration,
    probas=y_calibration_probas,
    groups=CALIBRATION_GROUPS,
    class_idx=CLASS_INDICES['DB'],
    target_recall=TARGET_RECALL,
    nms_window=NMS_WINDOW,
    tolerance=TOLERANCE,
)
THRESHOLD_SEARCH_RESULTS = {
    'Double Top (DT)': threshold_dt_result,
    'Double Bottom (DB)': threshold_db_result,
}
thresh_dt = threshold_dt_result['threshold']
thresh_db = threshold_db_result['threshold']

print('=' * 70)
print(f'TARGET EVENT RECALL: {TARGET_RECALL * 100:.1f}%')
for class_name, result in THRESHOLD_SEARCH_RESULTS.items():
    print(
        f"{class_name}: threshold={result['threshold']:.6f} | "
        f"status={result['status']} | "
        f"raw calibration Recall={result['raw_recall']:.4f} | "
        f"post-processed calibration Recall={result['event_recall']:.4f} | "
        f"post-processed FP events={result['event_FP']:,}"
    )
print('=' * 70)

# 3. Финальная validation-оценка использует зафиксированные calibration thresholds.
y_pred_raw = apply_pattern_thresholds(
    y_valid_probas,
    threshold_dt=thresh_dt,
    threshold_db=thresh_db,
)
y_pred_valid = apply_nms_clustering(
    y_pred_raw,
    y_valid_probas,
    window_size=NMS_WINDOW,
    groups=VALID_GROUPS,
)

tp_1, fp_1, fn_1 = get_tolerant_indices(
    y_valid,
    y_pred_valid,
    class_idx=CLASS_INDICES['DT'],
    tolerance=TOLERANCE,
    groups=VALID_GROUPS,
)
tp_2, fp_2, fn_2 = get_tolerant_indices(
    y_valid,
    y_pred_valid,
    class_idx=CLASS_INDICES['DB'],
    tolerance=TOLERANCE,
    groups=VALID_GROUPS,
)

event_dt = evaluate_event_level(
    y_true=y_valid,
    y_pred=y_pred_valid,
    probas=y_valid_probas,
    class_idx=CLASS_INDICES['DT'],
    groups=VALID_GROUPS,
    tolerance=TOLERANCE,
    event_gap=EVENT_GAP,
)
event_db = evaluate_event_level(
    y_true=y_valid,
    y_pred=y_pred_valid,
    probas=y_valid_probas,
    class_idx=CLASS_INDICES['DB'],
    groups=VALID_GROUPS,
    tolerance=TOLERANCE,
    event_gap=EVENT_GAP,
)
EVALUATION_RESULTS = {
    'Double Top (DT)': {
        'row_TP': len(tp_1),
        'row_FP': len(fp_1),
        'row_FN': len(fn_1),
        'event': event_dt,
    },
    'Double Bottom (DB)': {
        'row_TP': len(tp_2),
        'row_FP': len(fp_2),
        'row_FN': len(fn_2),
        'event': event_db,
    },
}

total_real_dt = count_physical_patterns(
    y_valid,
    class_idx=CLASS_INDICES['DT'],
    groups=VALID_GROUPS,
)
total_real_db = count_physical_patterns(
    y_valid,
    class_idx=CLASS_INDICES['DB'],
    groups=VALID_GROUPS,
)
caught_dt = total_real_dt - len(fn_1)
caught_db = total_real_db - len(fn_2)
real_recall_dt = caught_dt / total_real_dt if total_real_dt else 0.0
real_recall_db = caught_db / total_real_db if total_real_db else 0.0
total_signals_dt = len(tp_1) + len(fp_1)
total_signals_db = len(tp_2) + len(fp_2)
real_precision_dt = (
    len(tp_1) / total_signals_dt if total_signals_dt else 0.0
)
real_precision_db = (
    len(tp_2) / total_signals_db if total_signals_db else 0.0
)

candles_cnt = len(y_valid)
print(f'Validation candles: {candles_cnt:,}')
print(f'DT recall={real_recall_dt:.4f}, precision={real_precision_dt:.4f}')
print(f'DB recall={real_recall_db:.4f}, precision={real_precision_db:.4f}')

tn_dt = print_confusion_report(
    'Double Top (DT)',
    tp=len(tp_1),
    fp=len(fp_1),
    fn=len(fn_1),
    total_samples=candles_cnt,
)
tn_db = print_confusion_report(
    'Double Bottom (DB)',
    tp=len(tp_2),
    fp=len(fp_2),
    fn=len(fn_2),
    total_samples=candles_cnt,
)
print_event_level_report(event_dt, 'Double Top (DT)')
print_event_level_report(event_db, 'Double Bottom (DB)')

print('📊 Отрисовка PR-кривых...')
depict_pr_auc_custom_debug(
    y_valid,
    y_valid_probas,
    threshold_dt=thresh_dt,
    threshold_db=thresh_db,
)
depict_feature_importance(model, FEATURE_COLUMNS_NAME)


## FP budget diagnostics

In [ ]:
TARGET_FP_EVENTS = 3_000

print('\n=== FP BUDGET DIAGNOSTICS ===')
for class_name in ('Double Top (DT)', 'Double Bottom (DB)'):
    calibration_result = THRESHOLD_SEARCH_RESULTS[class_name]
    validation_result = EVALUATION_RESULTS[class_name]
    validation_event = validation_result['event']
    fp_delta = validation_event['FP'] - TARGET_FP_EVENTS
    recall = validation_event['Recall']
    status = (
        'PASS'
        if recall >= TARGET_RECALL
        and validation_event['FP'] <= TARGET_FP_EVENTS
        else 'CHECK'
    )
    print(
        f'{class_name}: {status} | '
        f'validation Recall={recall:.4f} | '
        f'validation FP events={validation_event["FP"]:,} '
        f'(delta={fp_delta:+,}) | '
        f'calibration FP events={calibration_result["event_FP"]:,}'
    )

combined_fp = sum(
    EVALUATION_RESULTS[class_name]['event']['FP']
    for class_name in ('Double Top (DT)', 'Double Bottom (DB)')
)
print(
    f'Combined validation FP events: {combined_fp:,} '
    f'(target approximately {2 * TARGET_FP_EVENTS:,})'
)


## NATR_14 diagnostics

In [ ]:
def audit_natr14_distribution(
    manifest,
    feature_name='NATR_14',
    sample_per_chunk=1_000,
):
    """Проверяет NATR_14 по split и labels, не собирая X в RAM."""
    feature_columns = list(manifest.feature_columns)
    feature_lookup = {
        str(column).lower(): index
        for index, column in enumerate(feature_columns)
    }
    feature_index = feature_lookup.get(feature_name.lower())
    if feature_index is None:
        print(
            f'⚠️ Feature {feature_name} отсутствует в manifest. '
            f'Первые доступные: {feature_columns[:10]}'
        )
        return {}

    stats = {}
    rng = np.random.default_rng(RANDOM_STATE)
    for split in ('train', 'calibration', 'validation'):
        split_stats = {
            class_name: {
                'count': 0,
                'finite': 0,
                'sum': 0.0,
                'sum_sq': 0.0,
                'min': np.inf,
                'max': -np.inf,
                'sample': [],
            }
            for class_name in ('NOISE', 'DT', 'DB')
        }
        for _, payload in batch_loader.iter_batches(split):
            X_chunk = payload['X']
            if hasattr(X_chunk, 'iloc'):
                values = np.asarray(X_chunk.iloc[:, feature_index], dtype=float)
            else:
                values = np.asarray(X_chunk[:, feature_index], dtype=float)
            labels = np.asarray(payload['y'])

            for class_name, class_idx in CLASS_INDICES.items():
                class_values = values[labels == class_idx]
                finite_values = class_values[np.isfinite(class_values)]
                current = split_stats[class_name]
                current['count'] += int(len(class_values))
                current['finite'] += int(len(finite_values))
                if len(finite_values):
                    current['sum'] += float(np.sum(finite_values))
                    current['sum_sq'] += float(
                        np.sum(np.square(finite_values))
                    )
                    current['min'] = min(
                        current['min'],
                        float(np.min(finite_values)),
                    )
                    current['max'] = max(
                        current['max'],
                        float(np.max(finite_values)),
                    )
                    sample_size = min(sample_per_chunk, len(finite_values))
                    sample_indices = rng.choice(
                        len(finite_values),
                        size=sample_size,
                        replace=False,
                    )
                    current['sample'].append(
                        finite_values[sample_indices]
                    )
            del X_chunk, values, labels, payload

        stats[split] = split_stats
        print(f'\n=== NATR_14 DISTRIBUTION: {split.upper()} ===')
        for class_name, current in split_stats.items():
            sample = (
                np.concatenate(current['sample'])
                if current['sample']
                else np.array([], dtype=float)
            )
            count = current['finite']
            mean = current['sum'] / count if count else np.nan
            variance = (
                current['sum_sq'] / count - mean ** 2
                if count
                else np.nan
            )
            std = np.sqrt(max(variance, 0.0)) if count else np.nan
            quantiles = (
                np.quantile(sample, [0.01, 0.50, 0.99])
                if len(sample)
                else [np.nan, np.nan, np.nan]
            )
            print(
                f'{class_name}: count={current["count"]:,}, '
                f'finite={count:,}, mean={mean:.6f}, std={std:.6f}, '
                f'q01={quantiles[0]:.6f}, median={quantiles[1]:.6f}, '
                f'q99={quantiles[2]:.6f}, '
                f'min={current["min"]:.6f}, max={current["max"]:.6f}'
            )
    return stats


NATR14_AUDIT = audit_natr14_distribution(batch_manifest)


## NATR_14 ablation

In [ ]:
RUN_NATR14_ABLATION = False

if not RUN_NATR14_ABLATION:
    print(
        'NATR_14 ablation не запускался. '
        'Установи RUN_NATR14_ABLATION=True и выполни эту ячейку '
        'для отдельного обучения без NATR_14.'
    )
else:
    if not isinstance(batch_loader, DatasetBatchLoader):
        raise RuntimeError(
            'NATR_14 ablation требует DatasetBatchLoader из batch pipeline.'
        )

    all_feature_columns = list(batch_manifest.feature_columns)
    ablation_feature_columns = [
        column for column in all_feature_columns
        if str(column).lower() != 'natr_14'
    ]
    if len(ablation_feature_columns) == len(all_feature_columns):
        raise ValueError('Feature NATR_14 не найден для ablation.')

    ablation_loader = DatasetBatchLoader(
        batch_manifest,
        feature_columns=ablation_feature_columns,
    )
    ablation_sampler = DatasetPatternSampler(
        noise_percent=CONFIG.noise_percent,
        dead_zone=CONFIG.dead_zone,
        random_state=RANDOM_STATE,
    )
    X_train_ablation, y_train_ablation, TRAIN_GROUPS_ABLATION = (
        ablation_sampler.build_train_arrays(ablation_loader)
    )
    model_without_natr14, ablation_cfg = auto_tune_xgboost_fbeta(
        X_train=X_train_ablation,
        y_train=y_train_ablation,
        n_iter=20,
        tolerance=CONFIG.tolerance,
        beta=2,
        train_is_sampled=True,
        train_groups=TRAIN_GROUPS_ABLATION,
    )
    y_cal_ablation, probas_cal_ablation, groups_cal_ablation = (
        ablation_loader.predict_split(
            model_without_natr14,
            'calibration',
        )
    )
    y_valid_ablation, probas_valid_ablation, groups_valid_ablation = (
        ablation_loader.predict_split(
            model_without_natr14,
            'validation',
        )
    )
    ablation_dt = select_postprocessed_threshold(
        y_cal_ablation,
        probas_cal_ablation,
        groups_cal_ablation,
        CLASS_INDICES['DT'],
        CONFIG.target_recall,
        CONFIG.nms_window,
        CONFIG.tolerance,
    )
    ablation_db = select_postprocessed_threshold(
        y_cal_ablation,
        probas_cal_ablation,
        groups_cal_ablation,
        CLASS_INDICES['DB'],
        CONFIG.target_recall,
        CONFIG.nms_window,
        CONFIG.tolerance,
    )
    print('NATR_14 ablation thresholds:', ablation_dt, ablation_db)
    del (
        X_train_ablation,
        y_train_ablation,
        probas_cal_ablation,
        probas_valid_ablation,
    )
    gc.collect()


## Confidence and PR diagnostics

In [ ]:
confidence_curves = plot_metrics_vs_confidence(
    y_true=y_valid,
    probas=y_valid_probas,
    class_indices=(CLASS_INDICES['DT'], CLASS_INDICES['DB']),
)


## Random false-positive visual audit

In [ ]:
def plot_random_false_positives(
    fp_indices,
    class_idx,
    class_name,
    num_charts=5,
    window_size=WINDOW_SIZE,
    random_state=None,
):
    """Рисует случайные FP в стиле SWP без загрузки всего raw dataset в RAM."""
    fp_indices = np.asarray(fp_indices, dtype=int)
    if len(fp_indices) == 0:
        print(f'Нет FP для {class_name}.')
        return

    rng = np.random.default_rng(random_state)
    chosen = rng.choice(
        fp_indices,
        size=min(num_charts, len(fp_indices)),
        replace=False,
    )
    class_names = {
        CLASS_INDICES['NOISE']: 'Noise',
        CLASS_INDICES['DT']: 'Double Top (DT)',
        CLASS_INDICES['DB']: 'Double Bottom (DB)',
    }
    frame_cache = {}
    fallback_labeled_dir = Path(
        '/content/drive/MyDrive/Crypto_Merged_Labeled_Data'
    )
    labeled_dir = Path(globals().get('LABELED_DATA_DIR', fallback_labeled_dir))
    padding = int(window_size * 0.33)

    print(
        f'=== Random FP audit: {class_name} '
        f'({len(chosen)} of {len(fp_indices)}) ==='
    )
    for chart_i, eval_idx in enumerate(chosen, start=1):
        group_id = int(VALID_GROUPS[eval_idx])
        if group_id >= len(symbols):
            print(f'Пропуск FP index={eval_idx}: неизвестный group_id={group_id}')
            continue
        symbol = symbols[group_id]

        # Position inside this symbol's validation split.
        local_validation_offset = int(
            np.sum(VALID_GROUPS[:eval_idx] == group_id)
        )
        train_rows = sum(
            record['rows']
            for record in batch_manifest.records_for('train', symbol)
        )
        calibration_rows = sum(
            record['rows']
            for record in batch_manifest.records_for('calibration', symbol)
        )
        global_position = (
            train_rows + calibration_rows + local_validation_offset
        )

        if symbol not in frame_cache:
            source_path = labeled_dir / (
                symbol.replace('/', '_') + '_labeled.csv'
            )
            if not source_path.exists():
                print(f'Пропуск {symbol}: файл не найден: {source_path}')
                continue
            frame_cache[symbol] = pd.read_csv(
                source_path,
                index_col=0,
                parse_dates=True,
            )
        frame = frame_cache[symbol]

        if global_position >= len(frame):
            print(
                f'Пропуск FP index={eval_idx}: '
                f'position={global_position}, rows={len(frame)}'
            )
            continue

        start_pos = max(0, global_position - window_size - padding)
        end_pos = min(len(frame), global_position + padding + 1)
        visible = frame.iloc[start_pos:end_pos].copy()
        x_values = np.arange(len(visible))
        trigger_x = global_position - start_pos
        pattern_start_x = trigger_x - window_size

        lower_columns = {str(column).lower(): column for column in visible.columns}
        has_ohlc = all(
            name in lower_columns
            for name in ('open', 'high', 'low', 'close')
        )
        fig, ax = plt.subplots(figsize=(12, 5))

        if has_ohlc:
            open_col = lower_columns['open']
            high_col = lower_columns['high']
            low_col = lower_columns['low']
            close_col = lower_columns['close']
            up_mask = visible[close_col] >= visible[open_col]
            down_mask = ~up_mask
            up = visible[up_mask]
            down = visible[down_mask]

            ax.vlines(
                x_values[up_mask.to_numpy()],
                up[low_col],
                up[high_col],
                color='#26A69A',
                linewidth=1.5,
            )
            ax.bar(
                x_values[up_mask.to_numpy()],
                up[close_col] - up[open_col],
                bottom=up[open_col],
                color='#26A69A',
                width=0.7,
            )
            ax.vlines(
                x_values[down_mask.to_numpy()],
                down[low_col],
                down[high_col],
                color='#EF5350',
                linewidth=1.5,
            )
            ax.bar(
                x_values[down_mask.to_numpy()],
                down[open_col] - down[close_col],
                bottom=down[close_col],
                color='#EF5350',
                width=0.7,
            )
            trigger_y = visible.iloc[trigger_x][close_col]
        else:
            numeric_columns = visible.select_dtypes(include='number').columns
            if len(numeric_columns) == 0:
                print(f'Пропуск {symbol}: нет числовых OHLC/price колонок')
                plt.close(fig)
                continue
            price_col = (
                'Close'
                if 'Close' in visible.columns
                else numeric_columns[0]
            )
            ax.plot(x_values, visible[price_col], color='black', linewidth=1.7)
            trigger_y = visible.iloc[trigger_x][price_col]

        ax.axvline(
            pattern_start_x,
            color='red',
            linestyle='--',
            linewidth=2,
            alpha=0.7,
        )
        ax.axvline(
            trigger_x,
            color='red',
            linestyle='--',
            linewidth=2,
            alpha=0.7,
        )
        ax.axvspan(
            pattern_start_x,
            trigger_x,
            color='red',
            alpha=0.05,
        )
        ax.scatter(
            trigger_x,
            trigger_y,
            color='blue',
            s=100,
            zorder=5,
            edgecolors='black',
            label='Сигнал алгоритма',
        )

        step = max(1, len(visible) // 10)
        tick_positions = x_values[::step]
        tick_labels = [
            visible.index[position].strftime('%m-%d %H:%M')
            if hasattr(visible.index[position], 'strftime')
            else str(visible.index[position])
            for position in tick_positions
        ]
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=15, ha='right', fontsize=9)

        probabilities = y_valid_probas[eval_idx]
        probability_text = '[' + ', '.join(
            f'{probability:.2f}' for probability in probabilities
        ) + ']'
        true_name = class_names.get(
            int(y_valid[eval_idx]),
            str(y_valid[eval_idx]),
        )
        predicted_name = class_names.get(class_idx, str(class_idx))
        ax.set_title(
            f'FP {class_name} | EvalIdx={eval_idx} | {symbol}\n'
            f'True={true_name} -> Pred={predicted_name}\n'
            f'Probs={probability_text}',
            fontsize=11,
        )
        ax.grid(alpha=0.3)
        ax.legend(loc='upper left')
        plt.tight_layout()
        plt.show()
        plt.close(fig)

    del frame_cache


In [ ]:
print('Визуальный аудит случайных False Positives:')
plot_random_false_positives(
    fp_indices=fp_1,
    class_idx=CLASS_INDICES['DT'],
    class_name='Double Top (DT)',
    num_charts=5,
    window_size=WINDOW_SIZE,
)
plot_random_false_positives(
    fp_indices=fp_2,
    class_idx=CLASS_INDICES['DB'],
    class_name='Double Bottom (DB)',
    num_charts=5,
    window_size=WINDOW_SIZE,
)


# Saving | loading model

## Constants for saving

In [55]:
model_name="xgb_dtdb_detector_v1.json"
base_dir="/content/drive/MyDrive/Crypto_pattern_detection_models"

## Saving model

In [56]:
def save_xgboost_model(model, model_name="xgb_dtdb_detector_v1.json",
                       base_dir="/content/drive/MyDrive/Crypto_pattern_detection_models"):
    """
    Сохраняет натренированную модель XGBoost на Google Drive.
    """
    # 1. Проверяем, примонтирован ли гугл диск (если нет - просим примонтировать)
    if not os.path.exists('/content/drive/MyDrive'):
        print("⚠️ Google Drive не примонтирован. Монтируем...")
        drive.mount('/content/drive')

    # 2. Создаем папку, если ее еще нет
    os.makedirs(base_dir, exist_ok=True)

    # 3. Формируем полный путь и сохраняем
    full_path = os.path.join(base_dir, model_name)
    model.save_model(full_path)

    print(f"✅ Модель успешно сохранена по пути:\n 📁 {full_path}")
    print(f"Размер файла: {os.path.getsize(full_path) / 1024:.2f} KB")

In [57]:
save_xgboost_model(model, model_name=model_name, base_dir=base_dir)

✅ Модель успешно сохранена по пути:
 📁 /content/drive/MyDrive/Crypto_pattern_detection_models/xgb_dtdb_detector_v1.json
Размер файла: 695.08 KB


## Loading model

In [ ]:
def load_xgboost_model(model_name="xgb_dtdb_detector_v1.json",
                       base_dir="/content/drive/MyDrive/Crypto_pattern_detection_models"):
    """
    Загружает модель XGBoost из Google Drive.
    """
    # 1. Проверяем доступ к диску
    if not os.path.exists('/content/drive/MyDrive'):
        print("⚠️ Google Drive не примонтирован. Монтируем...")
        drive.mount('/content/drive')

    full_path = os.path.join(base_dir, model_name)

    # 2. Проверяем, существует ли файл
    if not os.path.exists(full_path):
        raise FileNotFoundError(f"❌ Файл модели не найден: {full_path}")

    # 3. Инициализируем пустую модель и загружаем в нее веса
    print(f"⏳ Загрузка модели из:\n 📁 {full_path}...")
    loaded_model = xgb.XGBClassifier()
    loaded_model.load_model(full_path)

    print("✅ Модель успешно загружена и готова к предсказаниям!")
    return loaded_model

In [ ]:
model_name = globals().get(
    'model_name',
    'xgb_dtdb_detector_v1.json',
)
base_dir = globals().get(
    'base_dir',
    '/content/drive/MyDrive/Crypto_pattern_detection_models',
)
model = load_xgboost_model(model_name=model_name, base_dir=base_dir)

# Evaluation requires the persistent batch manifest in addition to model weights.
if 'BATCH_DRIVE_MANIFEST_PATH' not in globals():
    BATCH_DRIVE_MANIFEST_PATH = Path(
        '/content/drive/MyDrive/tickframe_batches/manifest.json'
    )

if 'CLASS_INDICES' not in globals():
    CLASS_INDICES = {'NOISE': 0, 'DT': 1, 'DB': 2}
TARGET_MAP = CLASS_INDICES

if 'FEATURE_COLUMNS_NAME' not in globals():
    FEATURE_COLUMNS_NAME = None

if 'CONFIG' not in globals():
    from types import SimpleNamespace
    CONFIG = SimpleNamespace(
        target_recall=0.75,
        tolerance=10,
        nms_window=10,
    )

if not BATCH_DRIVE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Batch manifest не найден: {BATCH_DRIVE_MANIFEST_PATH}. '
        'Сначала выполни сохранение tickframe_batches в Google Drive.'
    )

if 'DatasetBatchManifest' in globals() and 'DatasetBatchLoader' in globals():
    batch_manifest = DatasetBatchManifest.load(BATCH_DRIVE_MANIFEST_PATH)
    batch_loader = DatasetBatchLoader(batch_manifest)
else:
    batch_manifest, batch_loader = load_evaluation_batch_context(
        BATCH_DRIVE_MANIFEST_PATH
    )

if FEATURE_COLUMNS_NAME is None:
    FEATURE_COLUMNS_NAME = list(batch_manifest.feature_columns)
elif list(batch_manifest.feature_columns) != list(FEATURE_COLUMNS_NAME):
    raise ValueError(
        'Feature schema в Drive manifest не совпадает с FEATURE_COLUMNS_NAME'
    )

print('✅ Model и evaluation batch context загружены.')
